In [ ]:
from pathlib import Path
import os
import gc
import json
import random
import numpy as np
import xarray as xr
import torch

FORECAST_HOURS = int(os.environ.get("YANGTZE_LEAD", "1"))

# 两个阶段都使用同一设备；改此处可换卡或使用 cpu。
DEVICE = os.environ.get("YANGTZE_DEVICE", "cuda:0")
TIMESTEP = 6
DOWNNUM = 4
COVNUM = 2
TEST_SIZE = 0.2
VALID_SIZE = 0.1
FIRST_STAGE_SEED = 25
RESIDUAL_SEED = 25
IO_CHUNK = 128
MODEL_DIR = Path(os.environ.get("YANGTZE_MODEL_DIR", "models"))
RESULT_DIR = Path(os.environ.get("YANGTZE_RESULTS_DIR", "results"))

if isinstance(FORECAST_HOURS, bool) or not isinstance(FORECAST_HOURS, int) or not 1 <= FORECAST_HOURS <= 6:
    raise ValueError('FORECAST_HOURS 必须是 1–6 的整数。')
if DEVICE.startswith('cuda'):
    device_index = torch.device(DEVICE).index or 0
    if not torch.cuda.is_available() or device_index >= torch.cuda.device_count():
        raise RuntimeError(f'{DEVICE} 不可用，请核对当前内核和可见 GPU；可在本框修改 DEVICE。')
    torch.cuda.set_device(device_index)

BASE_MODEL_STEM = MODEL_DIR / f'DA_TransUNet_{FORECAST_HOURS}_24year_changjiang_vmd'
RESIDUAL_MODEL_STEM = MODEL_DIR / f'DA_TransUNet_{FORECAST_HOURS}_24year_changjiang_vmd_residual'
STAGE1_SCALER_PATH = Path(str(BASE_MODEL_STEM) + '_normalization.npz')
STAGE1_INFO_PATH = Path(str(BASE_MODEL_STEM) + '_run.json')
VMD_ALIGNMENT_PATH = Path(str(BASE_MODEL_STEM) + '_vmd_alignment.json')
SCALER_PATH = Path(str(RESIDUAL_MODEL_STEM) + '_normalization.npz')
RUN_INFO_PATH = Path(str(RESIDUAL_MODEL_STEM) + '_run.json')
COMPONENTS = {'u': slice(0, 3), 'v': slice(3, 6), 'w': slice(6, 9)}
LEVELS = np.asarray([850, 925, 1000], dtype=np.float32)

def nc_path(kind, component, suffix=''):
    if kind not in ('testy', 'predicty') or component not in COMPONENTS:
        raise ValueError('kind/component 不合法。')
    return RESULT_DIR / f'da_transunet_{kind}_{component}_{FORECAST_HOURS}_24year_changjiang_vmd{suffix}.nc'

def seed_residual_training(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# 与上传 notebook 的实际调用一致；仅模型加载/保存、split 模式由两个阶段分别指定。
COMMON_KWARGS = dict(
    test_size=TEST_SIZE, valid_size=VALID_SIZE, k_fold=None,
    task_mode='regression', if_bottom_attention_only='yes', if_resnet='yes',
    layer_time=1, baselayer=16, cov_kernelsize=3, cov_strides=1,
    pool_method='maxpooling', pool_kernel_size=2, pool_strides=2,
    ifnormalization='no', normalization_method=None,
    trans_num=1, num_heads=1, dim_feedforward=128, trans_dropout_rate=0.0,
    activate='tanh', nlp_activate='relu', if_last_act='no',
    if_print_model='yes', loss_function='default', optimizer='Adam',
    metrics='Pearsonr', learning_rate=0.0001, epochs=5000, batch_size=500,
    if_early_stopping=100, ifheatmap='no', ifweight='no', ifmute='no',
    device=DEVICE,
)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(f'预报时效：{FORECAST_HOURS} h；设备：{DEVICE}')
print(f'将训练并保存第一阶段：{BASE_MODEL_STEM}.pth')
print(f'残差模型：{RESIDUAL_MODEL_STEM}.pth')
print(f'结果目录：{RESULT_DIR}；第一阶段 _vmd；全集 _vmd_all；最终订正 _vmd_residual')


VMD_DIR = Path(os.environ.get("YANGTZE_VMD_DIR", str(Path(os.environ.get("YANGTZE_WORK_DIR","work")) / "vmd")))
VMD_SPECS = [
    dict(key='sp', stem='Surface-pressure', K=5),
    dict(key='t2m', stem='2m-temperature', K=6),
    dict(key='d2m', stem='2m-dewpoint-temperature', K=6),
]
VMD_SOURCE_NOTEBOOK = 'decompose.ipynb'
missing = [str(VMD_DIR / f"{s['stem']}_imf_{k}_{split}.nc")
           for s in VMD_SPECS for k in range(1, s['K']+1)
           for split in ('train', 'valid', 'test')
           if not (VMD_DIR / f"{s['stem']}_imf_{k}_{split}.nc").is_file()]
if missing:
    raise FileNotFoundError('缺少已完成的VMD文件，请核对 VMD_DIR：\n' + '\n'.join(missing))
print('VMD输入：SP 5 + T2M 6 + D2M 6 = 17个IMF；第一阶段20通道，第二阶段12通道。')


In [ ]:
#打开nc文件
def open_data_nc(ncmode,filename,v_name,iftime,timename,timestart,timeend,iflon,lonname,iflat,latname,latlow,lattop,lonleft,lonright,latresolution,lonresolution,ifexper,iflevel,levelname,level,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no'):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from netCDF4 import Dataset as net
    import xarray as xr
    from datetime import datetime,timedelta
    from dateutil.relativedelta import relativedelta
    import os
    
    plt.rcParams['font.sans-serif']=['SimHei'] #正常显示中文
    plt.rcParams['axes.unicode_minus']=False #正常显示正负号
    if ncmode == 'one':
        file = xr.open_dataset(filename)
        if ifinterpolate == 'yes':
            inter = str('file.interp('+latname+'=np.arange('+str(latlow)+','+str(lattop+latresolution)+','+str(latresolution)+'),'+lonname+'=np.arange('+str(lonleft)+','+str(lonright+lonresolution)+','+str(lonresolution)+'))')
            files=eval(inter)
            file = files
        if iftime  == 'yes' or iftime == 'self':
            times = np.array(file[timename])
        if iflon == 'yes':
            lon = np.array(file[lonname])
        if iflat == 'yes':
            lat = np.array(file[latname])
        v = file[v_name]
        if iflevel != 'no':
            levels = np.array(file[levelname])
    elif ncmode == 'more_time' or ncmode =='more_level':
        direc = os.listdir(filename)
        path = []
        file = []
        v = []
        lat = []
        lon = []
        times = []
        levels = []
        for i in range(len(direc)):
            if filename[-1] == '/':  
                path.append(filename+str(direc[i]))
            else:
                path.append(filename+'/'+str(direc[i]))
            file_xr = xr.open_dataset(path[i])
            if ifinterpolate == 'yes':
                inter = str('file_xr.interp('+latname+'=np.arange('+str(latlow)+','+str(lattop)+','+str(latresolution)+'),'+lonname+'=np.arange('+str(lonleft)+','+str(lonright)+','+str(lonresolution)+'))')
                files=eval(inter)
                file_xr = files
            file.append(file_xr)
            if ncmode == 'more_time':
                vs=np.array(file[i][v_name])
                if iftime =='yes':
                    timelist=np.array(file[i][timename])
                if i != 0:
                    if iftime =='yes':
                        v=np.concatenate((v,vs))
                        times=np.concatenate((times,timelist))
                    elif iftime =='create':
                        if iflevel !='no':
                            if iflat !='no':
                                if iflon !='no':
                                    vs = vs.reshape((1,vs.shape[0],vs.shape[1],vs.shape[2]))
                                else:
                                    vs = vs.reshape((1,vs.shape[0],vs.shape[1]))
                            else:
                                if iflon !='no':
                                    vs = vs.reshape((1,vs.shape[0],vs.shape[1]))
                                else:
                                    vs = vs.reshape((1,vs.shape[0]))
                        else:
                            if iflat !='no':
                                if iflon !='no':
                                    vs = vs.reshape((1,vs.shape[0],vs.shape[1]))
                                else:
                                    vs = vs.reshape((1,vs.shape[0]))
                            else:
                                if iflon !='no':
                                    vs = vs.reshape((1,vs.shape[0]))
                                else:
                                    vs = vs.reshape((1))
                        v=np.concatenate((v,vs))
                else:
                    if iftime == 'create':
                        if iflevel !='no':
                            if iflat !='no':
                                if iflon !='no':
                                    vs = vs.reshape((1,vs.shape[0],vs.shape[1],vs.shape[2]))
                                else:
                                    vs = vs.reshape((1,vs.shape[0],vs.shape[1]))
                            else:
                                if iflon !='no':
                                    vs = vs.reshape((1,vs.shape[0],vs.shape[1]))
                                else:
                                    vs = vs.reshape((1,vs.shape[0]))
                        else:
                            if iflat !='no':
                                if iflon !='no':
                                    vs = vs.reshape((1,vs.shape[0],vs.shape[1]))
                                else:
                                    vs = vs.reshape((1,vs.shape[0]))
                            else:
                                if iflon !='no':
                                    vs = vs.reshape((1,vs.shape[0]))
                                else:
                                    vs = vs.reshape((1))
                        v=vs
                    elif iftime == 'yes':
                        v = vs
                        times=timelist
            if ncmode == 'more_level':
                if iflevel == 'create':
                    vs=np.array(file[i][v_name])
                    levels=level
                elif iflevel == 'yes' or iflevel =='all' or iflevel =='self' or iflevel =='selfchose':
                    if iftime !='no':
                        if iflat !='no':
                            if iflon !='no':
                                vs=np.array(file[i][v_name]).transpose(1,0,2,3)
                            else:
                                vs=np.array(file[i][v_name]).transpose(1,0,2)
                        else:
                            if iflon !='no':
                                vs=np.array(file[i][v_name]).transpose(1,0,2)
                            else:
                                vs=np.array(file[i][v_name]).transpose(1,0)
                    levellist=np.array(file[i][levelname])      
                if i != 0:
                    if iflevel == 'yes' or iflevel =='all' or iflevel =='self' or iflevel =='selfchose':
                        v=np.concatenate((v,vs))
                        levels=np.concatenate((levels,levellist))
                    elif iflevel =='create':
                        if iftime !='no':
                            if iflat !='no':
                                if iflon !='no':
                                    vs = vs.reshape((1,vs.shape[0],vs.shape[1],vs.shape[2]))
                                else:
                                    vs = vs.reshape((1,vs.shape[0],vs.shape[1]))
                            else:
                                if iflon !='no':
                                    vs = vs.reshape((1,vs.shape[0],vs.shape[1]))
                                else:
                                    vs = vs.reshape((1,vs.shape[0]))
                        else:
                            if iflat !='no':
                                if iflon !='no':
                                    vs = vs.reshape((1,vs.shape[0],vs.shape[1]))
                                else:
                                    vs = vs.reshape((1,vs.shape[0]))
                            else:
                                if iflon !='no':
                                    vs = vs.reshape((1,vs.shape[0]))
                                else:
                                    vs = vs.reshape((1))
                        v=np.concatenate((v,vs))
                else:
                    if iflevel == 'create':
                        if iftime !='no':
                            if iflat !='no':
                                if iflon !='no':
                                    vs = vs.reshape((1,vs.shape[0],vs.shape[1],vs.shape[2]))
                                else:
                                    vs = vs.reshape((1,vs.shape[0],vs.shape[1]))
                            else:
                                if iflon !='no':
                                    vs = vs.reshape((1,vs.shape[0],vs.shape[1]))
                                else:
                                    vs = vs.reshape((1,vs.shape[0]))
                        else:
                            if iflat !='no':
                                if iflon !='no':
                                    vs = vs.reshape((1,vs.shape[0],vs.shape[1]))
                                else:
                                    vs = vs.reshape((1,vs.shape[0]))
                            else:
                                if iflon !='no':
                                    vs = vs.reshape((1,vs.shape[0]))
                                else:
                                    vs = vs.reshape((1))
                        v=vs
                    elif iflevel == 'yes' or iflevel =='all' or iflevel =='self' or iflevel =='selfchose':
                        v=vs
                        levels=levellist
        if ncmode == 'more_time':
            if iflon =='yes':
                lon = file[0][lonname]
            if iflat =='yes':
                lat = file[0][latname]
            if iflevel != 'no':
                levels = np.array(file[0][levelname])
        if ncmode == 'more_level':
            if iflon =='yes':
                lon = file[0][lonname]
            if iflat =='yes':
                lat = file[0][latname]
            if iftime != 'no':
                times = np.array(file[0][timename])
            if iftime !='no':
                if iflat !='no':
                    if iflon !='no':
                        v=v.transpose(1,0,2,3)
                    else:
                        v=v.transpose(1,0,2)
                else:
                    if iflon !='no':
                        v=v.transpose(1,0,2)
                    else:
                        v=v.transpose(1,0)
    elif ncmode == 'one_wrf':
        file = xr.open_dataset(filename)
        ncfile = net(filename)
        times = np.array(file[timename])
        lon = np.array(file[lonname][0,0,:])
        lat = np.array(file[latname][0,:,0])
        if iflevel == 'no':
            v = np.zeros((times.shape[0],lat.shape[0],lon.shape[0]))
            for i in range(times.shape[0]):
                v[i,:,:] = np.array(getvar(ncfile,v_name,i))
        elif iflevel == 'yes':
            levels = np.array(file[levelname])[0,:]
            p = np.zeros((times.shape[0],levels.shape[0],lat.shape[0],lon.shape[0]))
            v = np.zeros((times.shape[0],levels.shape[0],lat.shape[0],lon.shape[0]))
            for i in range(times.shape[0]):
                if v_name == 'U':
                    v[i,:,:,:] = np.array(getvar(ncfile,v_name,i))[:,:,:-1]
                elif v_name == 'V':
                    v[i,:,:,:] = np.array(getvar(ncfile,v_name,i))[:,:-1,:]
                elif v_name == 'W' or v_name == 'PH' or v_name == 'PHB':
                    v[i,:,:,:] = np.array(getvar(ncfile,v_name,i))[:-1,:,:]
                else:
                    v[i,:,:,:] = np.array(getvar(ncfile,v_name,i))
                p[i,:,:,:] = np.array(getvar(ncfile,'pressure',i))
            vs = np.zeros((times.shape[0],lat.shape[0],lon.shape[0]))
            for i in range(times.shape[0]):
                vs[i,:,:] = interplevel(v[i,:,:,:],p[i,:,:,:],level)
        else:
            levels = np.array(file[levelname])[0,:]
            p = np.zeros((times.shape[0],levels.shape[0],lat.shape[0],lon.shape[0]))
            v = np.zeros((times.shape[0],levels.shape[0],lat.shape[0],lon.shape[0]))
            for i in range(times.shape[0]):
                if v_name == 'U':
                    v[i,:,:,:] = np.array(getvar(ncfile,v_name,i))[:,:,:-1]
                elif v_name == 'V':
                    v[i,:,:,:] = np.array(getvar(ncfile,v_name,i))[:,:-1,:]
                elif v_name == 'W' or v_name == 'PH' or v_name == 'PHB':
                    v[i,:,:,:] = np.array(getvar(ncfile,v_name,i))[:-1,:,:]
                else:
                    v[i,:,:,:] = np.array(getvar(ncfile,v_name,i))
                p[i,:,:,:] = np.array(getvar(ncfile,'pressure',i))
            vs = np.zeros((times.shape[0],len(level),lat.shape[0],lon.shape[0]))
            for i in range(times.shape[0]):
                vs[i,:,:,:] = interplevel(v[i,:,:,:],p[i,:,:,:],level)
        if iflevel !='no':
            levels = level
            v = vs
    if iftime =='yes' or iftime == 'create':
        if len(timestart) == 4 :
            if iftime =='yes':
                for i in range(len(times)):
                    if timestart == pd.to_datetime(str(np.array(times[i]))).strftime('%Y'):
                        startpoint = i
                    if timeend == pd.to_datetime(str(np.array(times[i]))).strftime('%Y'):
                        endpoint = i
            if iftime =='create':
                for i in range(v.shape[0]):
                    times.append(datetime(int(pd.to_datetime(str(timestart)).strftime('%Y')),int(pd.to_datetime(str(timestart)).strftime('%m')),int(pd.to_datetime(str(timestart)).strftime('%d')))+ timespace*i * relativedelta(years=+1))
                    times[i]=pd.to_datetime(str(times[i])).strftime('%Y-%m-%d')
                times = np.array(times,dtype = np.datetime64)
        elif len(timestart) == 7 :
            if iftime =='yes':
                for i in range(len(times)):
                    if timestart == pd.to_datetime(str(np.array(times[i]))).strftime('%Y-%m'):
                        startpoint = i
                    if timeend == pd.to_datetime(str(np.array(times[i]))).strftime('%Y-%m'):
                        endpoint = i
            if iftime =='create':
                for i in range(v.shape[0]):
                    times.append(datetime(int(pd.to_datetime(str(timestart)).strftime('%Y')),int(pd.to_datetime(str(timestart)).strftime('%m')),int(pd.to_datetime(str(timestart)).strftime('%d')))+ timespace*i * relativedelta(months=+1))
                    times[i]=pd.to_datetime(str(times[i])).strftime('%Y-%m-%d')
                times = np.array(times,dtype = np.datetime64)
        elif len(timestart) == 10 :
            if iftime =='yes':
                for i in range(len(times)):
                    if timestart == pd.to_datetime(str(np.array(times[i]))).strftime('%Y-%m-%d'):
                        startpoint = i
                    if timeend == pd.to_datetime(str(np.array(times[i]))).strftime('%Y-%m-%d'):
                        endpoint = i
            if iftime =='create':
                for i in range(v.shape[0]):
                    times.append(datetime(int(pd.to_datetime(str(timestart)).strftime('%Y')),int(pd.to_datetime(str(timestart)).strftime('%m')),int(pd.to_datetime(str(timestart)).strftime('%d')))+ timespace*i * timedelta(days=1))
                    times[i]=pd.to_datetime(str(times[i])).strftime('%Y-%m-%d')
                times = np.array(times,dtype = np.datetime64)
        elif len(timestart) == 13 :
            if iftime =='yes':
                for i in range(len(times)):
                    if timestart == pd.to_datetime(str(np.array(times[i]))).strftime('%Y-%m-%d-%H'):
                        startpoint = i
                    if timeend == pd.to_datetime(str(np.array(times[i]))).strftime('%Y-%m-%d-%H'):
                        endpoint = i
            if iftime =='create':
                for i in range(v.shape[0]):
                    times.append(datetime(int(pd.to_datetime(str(timestart)).strftime('%Y')),int(pd.to_datetime(str(timestart)).strftime('%m')),int(pd.to_datetime(str(timestart)).strftime('%d')),int(pd.to_datetime(str(timestart)).strftime('%H')))+ timespace*i * timedelta(hours=1))
                    times[i]=pd.to_datetime(str(times[i])).strftime('%Y-%m-%d %H:%M:%S')
                times = np.array(times,dtype = np.datetime64)
        elif len(timestart) == 16 :
            if iftime =='yes':
                for i in range(len(times)):
                    if timestart == pd.to_datetime(str(np.array(times[i]))).strftime('%Y-%m-%d-%H-%M'):
                        startpoint = i
                    if timeend == pd.to_datetime(str(np.array(times[i]))).strftime('%Y-%m-%d-%H-%M'):
                        endpoint = i
            if iftime =='create':
                for i in range(v.shape[0]):
                    times.append(datetime(int(pd.to_datetime(str(timestart)).strftime('%Y')),int(pd.to_datetime(str(timestart)).strftime('%m')),int(pd.to_datetime(str(timestart)).strftime('%d')),int(pd.to_datetime(str(timestart)).strftime('%H')),int(pd.to_datetime(str(timestart)).strftime('%M')))+ timespace*i * timedelta(minutes=1))
                    times[i]=pd.to_datetime(str(times[i])).strftime('%Y-%m-%d %H:%M:%S')
                times = np.array(times,dtype = np.datetime64)
        elif len(timestart) == 19 :
            if iftime =='yes':
                for i in range(len(times)):
                    if timestart == pd.to_datetime(str(np.array(times[i]))).strftime('%Y-%m-%d-%H-%M-%S'):
                        startpoint = i
                    if timeend == pd.to_datetime(str(np.array(times[i]))).strftime('%Y-%m-%d-%H-%M-%S'):
                        endpoint = i
            if iftime =='create':
                for i in range(v.shape[0]):
                    times.append(datetime(int(pd.to_datetime(str(timestart)).strftime('%Y')),int(pd.to_datetime(str(timestart)).strftime('%m')),int(pd.to_datetime(str(timestart)).strftime('%d')),int(pd.to_datetime(str(timestart)).strftime('%H')),int(pd.to_datetime(str(timestart)).strftime('%M')),int(pd.to_datetime(str(timestart)).strftime('%S')))+ timespace*i * timedelta(seconds=1))
                    times[i]=pd.to_datetime(str(times[i])).strftime('%Y-%m-%d %H:%M:%S')
                times = np.array(times,dtype = np.datetime64)
    if iftime=='self':
        for i in range(len(times)):
            if timestart == times[i]:
                startpoint = i
            if timeend == times[i]:
                endpoint = i
    if iftime =='yes' or iftime=='self':
        times = times[startpoint:endpoint+1]
    elif iftime =='create':
        startpoint = 0
        endpoint = times.shape[0]
    if iflat == 'yes':
        if float(lat[0])>float(lat[1]):
            lowpoint = int((np.nanmax(lat)-latlow)/latresolution)
            toppoint = int((np.nanmax(lat)-lattop)/latresolution)
        else:
            lowpoint = int((-np.nanmin(lat)+latlow)/latresolution)
            toppoint = int((-np.nanmin(lat)+lattop)/latresolution)
    if iflon == 'yes':
        leftpoint = int((-np.nanmin(lon)+lonleft)/lonresolution)
        rightpoint = int((-np.nanmin(lon)+lonright)/lonresolution)
    if ncmode != 'one_wrf':
        if iflevel == 'yes':
            for i in range(0,len(levels)):
                if int(level) == int(levels[i]):
                    levelpoint = i
            if ifexper == 'yes':
                if float(lat[0])>float(lat[1]):
                    v = v[startpoint:endpoint+1,0,levelpoint,toppoint:lowpoint+1,leftpoint:rightpoint+1]
                    v = np.array(v[:,::changeresolution,::changeresolution])
                else:
                    v = v[startpoint:endpoint+1,0,levelpoint,lowpoint:toppoint+1,leftpoint:rightpoint+1]
                    v = np.array(v[:,::changeresolution,::changeresolution])
            elif ifexper ==  'no':
                if iftime != 'no':
                    if iflon != 'no':
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[startpoint:endpoint+1,levelpoint,toppoint:lowpoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[:,::changeresolution,::changeresolution])
                            else:
                                v = v[startpoint:endpoint+1,levelpoint,lowpoint:toppoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[:,::changeresolution,::changeresolution])
                        else:
                            v = v[startpoint:endpoint+1,levelpoint,leftpoint:rightpoint+1]
                            v = np.array(v[:,::changeresolution])
                    else:
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[startpoint:endpoint+1,levelpoint,toppoint:lowpoint+1]
                                v = np.array(v[:,::changeresolution])
                            else:
                                v = v[startpoint:endpoint+1,levelpoint,lowpoint:toppoint+1]
                                v = np.array(v[:,::changeresolution])
                        else:
                            v = v[startpoint:endpoint+1,levelpoint]
                            v = np.array(v)
                else:
                    if iflon != 'no':
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[levelpoint,toppoint:lowpoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[::changeresolution,::changeresolution])
                            else:
                                v = v[levelpoint,lowpoint:toppoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[::changeresolution,::changeresolution])
                        else:
                            v = v[levelpoint,leftpoint:rightpoint+1]
                            v = np.array(v[::changeresolution])
                    else:
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[levelpoint,toppoint:lowpoint+1]
                                v = np.array(v[::changeresolution])
                            else:
                                v = v[levelpoint,lowpoint:toppoint+1]
                                v = np.array(v[::changeresolution])
                        else:
                            v = v[levelpoint]
                            v = np.array(v)
        elif iflevel == 'no':
            if ifexper == 'yes':
                if float(lat[0])>float(lat[1]):
                    v = v[startpoint:endpoint+1,0,toppoint:lowpoint+1,leftpoint:rightpoint+1]
                    v = np.array(v[:,::changeresolution,::changeresolution])
                else:
                    v = v[startpoint:endpoint+1,0,lowpoint:toppoint+1,leftpoint:rightpoint+1]
                    v = np.array(v[:,::changeresolution,::changeresolution])
            elif ifexper ==  'no':
                if iftime != 'no':
                    if iflon != 'no':
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[startpoint:endpoint+1,toppoint:lowpoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[:,::changeresolution,::changeresolution])
                            else:
                                v = v[startpoint:endpoint+1,lowpoint:toppoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[:,::changeresolution,::changeresolution])
                        else:
                            v = v[startpoint:endpoint+1,leftpoint:rightpoint+1]
                            v = np.array(v[:,::changeresolution])
                    else:
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[startpoint:endpoint+1,toppoint:lowpoint+1]
                                v = np.array(v[:,::changeresolution])
                            else:
                                v = v[startpoint:endpoint+1,lowpoint:toppoint+1]
                                v = np.array(v[:,::changeresolution])
                        else:
                            v = v[startpoint:endpoint+1]
                            v = np.array(v)
                else:
                    if iflon != 'no':
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[toppoint:lowpoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[::changeresolution,::changeresolution])
                            else:
                                v = v[lowpoint:toppoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[::changeresolution,::changeresolution])
                        else:
                            v = v[leftpoint:rightpoint+1]
                            v = np.array(v[::changeresolution])
                    else:
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[toppoint:lowpoint+1]
                                v = np.array(v[::changeresolution])
                            else:
                                v = v[lowpoint:toppoint+1]
                                v = np.array(v[::changeresolution])
                        else:
                            v = None
        elif iflevel == 'all' or iflevel =='create':
            if ifexper == 'yes':
                if float(lat[0])>float(lat[1]):
                    v = v[startpoint:endpoint+1,0,:,toppoint:lowpoint+1,leftpoint:rightpoint+1]
                    v = np.array(v[:,:,::changeresolution,::changeresolution])
                else:
                    v = v[startpoint:endpoint+1,0,:,lowpoint:toppoint+1,leftpoint:rightpoint+1]
                    v = np.array(v[:,:,::changeresolution,::changeresolution])
            elif ifexper == 'no':
                if iftime != 'no':
                    if iflon != 'no':
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[startpoint:endpoint+1,:,toppoint:lowpoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[:,:,::changeresolution,::changeresolution])
                            else:
                                v = v[startpoint:endpoint+1,:,lowpoint:toppoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[:,:,::changeresolution,::changeresolution])
                        else:
                            v = v[startpoint:endpoint+1,:,leftpoint:rightpoint+1]
                            v = np.array(v[:,:,::changeresolution])
                    else:
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[startpoint:endpoint+1,:,toppoint:lowpoint+1]
                                v = np.array(v[:,:,::changeresolution])
                            else:
                                v = v[startpoint:endpoint+1,:,lowpoint:toppoint+1]
                                v = np.array(v[:,:,::changeresolution])
                        else:
                            v = v[startpoint:endpoint+1,:]
                            v = np.array(v)
                else:
                    if iflon != 'no':
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[:,toppoint:lowpoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[:,::changeresolution,::changeresolution])
                            else:
                                v = v[:,lowpoint:toppoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[:,::changeresolution,::changeresolution])
                        else:
                            v = v[:,leftpoint:rightpoint+1]
                            v = np.array(v[:,::changeresolution])
                    else:
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[:,toppoint:lowpoint+1]
                                v = np.array(v[:,::changeresolution])
                            else:
                                v = v[:,lowpoint:toppoint+1]
                                v = np.array(v[:,::changeresolution])
                        else:
                            v = v[:]
                            v = np.array(v)
        elif iflevel == 'self':
            levelstart = 0
            levelend = 0
            for i in range(len(levels)):
                if int(levels[i]) == level[0]:
                    levelstart = i
                if int(levels[i]) == level[1]:
                    levelend = i
            if ifexper == 'yes':
                if float(lat[0])>float(lat[1]):
                    v = v[startpoint:endpoint+1,0,levelstart:levelend+1,toppoint:lowpoint+1,leftpoint:rightpoint+1]
                    v = np.array(v[:,:,::changeresolution,::changeresolution])
                else:
                    v = v[startpoint:endpoint+1,0,levelstart:levelend+1,lowpoint:toppoint+1,leftpoint:rightpoint+1]
                    v = np.array(v[:,:,::changeresolution,::changeresolution])
            elif ifexper == 'no':
                if iftime != 'no':
                    if iflon != 'no':
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[startpoint:endpoint+1,levelstart:levelend+1,toppoint:lowpoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[:,:,::changeresolution,::changeresolution])
                            else:
                                v = v[startpoint:endpoint+1,levelstart:levelend+1,lowpoint:toppoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[:,:,::changeresolution,::changeresolution])
                        else:
                            v = v[startpoint:endpoint+1,levelstart:levelend+1,leftpoint:rightpoint+1]
                            v = np.array(v[:,:,::changeresolution])
                    else:
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[startpoint:endpoint+1,levelstart:levelend+1,toppoint:lowpoint+1]
                                v = np.array(v[:,:,::changeresolution])
                            else:
                                v = v[startpoint:endpoint+1,levelstart:levelend+1,lowpoint:toppoint+1]
                                v = np.array(v[:,:,::changeresolution])
                        else:
                            v = v[startpoint:endpoint+1,levelstart:levelend+1]
                            v = np.array(v)
                else:
                    if iflon != 'no':
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[levelstart:levelend+1,toppoint:lowpoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[:,::changeresolution,::changeresolution])
                            else:
                                v = v[levelstart:levelend+1,lowpoint:toppoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[:,::changeresolution,::changeresolution])
                        else:
                            v = v[levelstart:levelend+1,leftpoint:rightpoint+1]
                            v = np.array(v[:,::changeresolution])
                    else:
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[levelstart:levelend+1,toppoint:lowpoint+1]
                                v = np.array(v[:,::changeresolution])
                            else:
                                v = v[levelstart:levelend+1,lowpoint:toppoint+1]
                                v = np.array(v[:,::changeresolution])
                        else:
                            v = v[levelstart:levelend+1]
                            v = np.array(v)
            levels = levels[levelstart:levelend+1]
        elif iflevel == 'selfchose':
            selflevel = []
            j=0
            for i in range(len(levels)):
                if j>= len(level):
                    break
                if int(levels[i]) == level[j]:
                    selflevel.append(i)
                    j=j+1
            if ifexper == 'yes':
                if float(lat[0])>float(lat[1]):
                    v = v[startpoint:endpoint+1,0,selflevel,toppoint:lowpoint+1,leftpoint:rightpoint+1]
                    v = np.array(v[:,:,::changeresolution,::changeresolution])
                else:
                    v = v[startpoint:endpoint+1,0,selflevel,lowpoint:toppoint+1,leftpoint:rightpoint+1]
                    v = np.array(v[:,:,::changeresolution,::changeresolution])
            elif ifexper == 'no':
                if iftime != 'no':
                    if iflon != 'no':
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[startpoint:endpoint+1,selflevel,toppoint:lowpoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[:,:,::changeresolution,::changeresolution])
                            else:
                                v = v[startpoint:endpoint+1,selflevel,lowpoint:toppoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[:,:,::changeresolution,::changeresolution])
                        else:
                            v = v[startpoint:endpoint+1,selflevel,leftpoint:rightpoint+1]
                            v = np.array(v[:,:,::changeresolution])
                    else:
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[startpoint:endpoint+1,selflevel,toppoint:lowpoint+1]
                                v = np.array(v[:,:,::changeresolution])
                            else:
                                v = v[startpoint:endpoint+1,selflevel,lowpoint:toppoint+1]
                                v = np.array(v[:,:,::changeresolution])
                        else:
                            v = v[startpoint:endpoint+1,selflevel]
                            v = np.array(v)
                else:
                    if iflon != 'no':
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[selflevel,toppoint:lowpoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[:,::changeresolution,::changeresolution])
                            else:
                                v = v[selflevel,lowpoint:toppoint+1,leftpoint:rightpoint+1]
                                v = np.array(v[:,::changeresolution,::changeresolution])
                        else:
                            v = v[selflevel,leftpoint:rightpoint+1]
                            v = np.array(v[:,::changeresolution])
                    else:
                        if iflat != 'no':
                            if float(lat[0])>float(lat[1]):
                                v = v[selflevel,toppoint:lowpoint+1]
                                v = np.array(v[:,::changeresolution])
                            else:
                                v = v[selflevel,lowpoint:toppoint+1]
                                v = np.array(v[:,::changeresolution])
                        else:
                            v = v[selflevel]
                            v = np.array(v)
            levels = levels[selflevel]
    else:
        if iflevel == 'yes' or iflevel == 'no':
            if float(lat[0])>float(lat[1]):
                v = v[startpoint:endpoint+1,toppoint:lowpoint+1,leftpoint:rightpoint+1]
                v = np.array(v[:,::changeresolution,::changeresolution])
            else:
                v = v[startpoint:endpoint+1,lowpoint:toppoint+1,leftpoint:rightpoint+1]
                v = np.array(v[:,::changeresolution,::changeresolution])
        else:
            if float(lat[0])>float(lat[1]):
                v = v[startpoint:endpoint+1,:,toppoint:lowpoint+1,leftpoint:rightpoint+1]
                v = np.array(v[:,:,::changeresolution,::changeresolution])
            else:
                v = v[startpoint:endpoint+1,:,lowpoint:toppoint+1,leftpoint:rightpoint+1]
                v = np.array(v[:,:,::changeresolution,::changeresolution])
    if iflon !='no':
        lon = lon[leftpoint:rightpoint+1:changeresolution]
    if iflat !='no':
        if float(lat[0])>float(lat[1]):
            lat = lat[toppoint:lowpoint+1:changeresolution]
        else:
            lat = lat[lowpoint:toppoint+1:changeresolution]
    if ifchange_west_east =='yes':
        if np.nanmin(lon)<0:
            right = 360.0 - changeresolution*lonresolution
            if iflevel == 'all' or iflevel == 'self' or iflevel == 'selfchose' or iflevel == 'create':
                if iftime !='no':
                    mid = int(v.shape[3]/2)
                    lon = np.linspace(0.0,right,v.shape[3])
                    vwest = v[:,:,:,0:mid]
                    veast = v[:,:,:,mid:]
                    v = np.concatenate((veast,vwest),axis=3)
                    lonleft = 0.0
                    lonright = right
                else:
                    mid = int(v.shape[2]/2)
                    lon = np.linspace(0.0,right,v.shape[2])
                    vwest = v[:,:,0:mid]
                    veast = v[:,:,mid:]
                    v = np.concatenate((veast,vwest),axis=2)
                    lonleft = 0.0
                    lonright = right
            else:
                if iftime !='no':
                    mid = int(v.shape[2]/2)
                    lon = np.linspace(0.0,right,v.shape[2])
                    vwest = v[:,:,0:mid]
                    veast = v[:,:,mid:]
                    v = np.concatenate((veast,vwest),axis=2)
                    lonleft = 0.0
                    lonright = right
                else:
                    mid = int(v.shape[1]/2)
                    lon = np.linspace(0.0,right,v.shape[1])
                    vwest = v[:,0:mid]
                    veast = v[:,mid:]
                    v = np.concatenate((veast,vwest),axis=1)
                    lonleft = 0.0
                    lonright = right
        else:
            right = 180.0 - changeresolution*lonresolution
            if iflevel == 'all' or iflevel == 'self' or iflevel == 'selfchose' or iflevel =='create':
                if iftime !='no':
                    mid = int(v.shape[3]/2)
                    lon = np.linspace(-180.0,right,v.shape[3])
                    veast = v[:,:,:,0:mid]
                    vwest = v[:,:,:,mid:]
                    v = np.concatenate((vwest,veast),axis=3)
                    lonleft = -180.0
                    lonright = right
                else:
                    mid = int(v.shape[2]/2)
                    lon = np.linspace(-180.0,right,v.shape[2])
                    veast = v[:,:,0:mid]
                    vwest = v[:,:,mid:]
                    v = np.concatenate((vwest,veast),axis=2)
                    lonleft = -180.0
                    lonright = right
            else:
                if iftime !='no':
                    mid = int(v.shape[2]/2)
                    lon = np.linspace(-180.0,right,v.shape[2])
                    veast = v[:,:,0:mid]
                    vwest = v[:,:,mid:]
                    v = np.concatenate((vwest,veast),axis=2)
                    lonleft = -180.0
                    lonright = right
                else:
                    mid = int(v.shape[1]/2)
                    lon = np.linspace(-180.0,right,v.shape[1])
                    veast = v[:,0:mid]
                    vwest = v[:,mid:]
                    v = np.concatenate((vwest,veast),axis=1)
                    lonleft = -180.0
                    lonright = right
    if iflevel == 'all' or iflevel == 'self' or iflevel == 'selfchose' or iflevel =='create':
        if iftime !='no':
            if iflat !='no':
                if iflon !='no':
                    v = xr.DataArray(v, [(timename,times),(levelname,levels),(latname,lat),(lonname,lon)])
                else:
                    v = xr.DataArray(v, [(timename,times),(levelname,levels),(latname,lat)])
            else:
                if iflon !='no':
                    v = xr.DataArray(v, [(timename,times),(levelname,levels),(lonname,lon)])
                else:
                    v = xr.DataArray(v, [(timename,times),(levelname,levels)])
        else:
            if iflat !='no':
                if iflon !='no':
                    v = xr.DataArray(v, [(levelname,levels),(latname,lat),(lonname,lon)])
                else:
                    v = xr.DataArray(v, [(levelname,levels),(latname,lat)])
            else:
                if iflon !='no':
                    v = xr.DataArray(v, [(levelname,levels),(lonname,lon)])
                else:
                    v = xr.DataArray(v, [(levelname,levels)])
        levels = v[levelname]
    else:
        if iftime !='no':
            if iflat !='no':
                if iflon !='no':
                    v = xr.DataArray(v, [(timename,times),(latname,lat),(lonname,lon)])
                else:
                    v = xr.DataArray(v, [(timename,times),(latname,lat)])
            else:
                if iflon !='no':
                    v = xr.DataArray(v, [(timename,times),(lonname,lon)])
                else:
                    v = xr.DataArray(v, [(timename,times)])
        else:
            if iflat !='no':
                if iflon !='no':
                    v = xr.DataArray(v, [(latname,lat),(lonname,lon)])
                else:
                    v = xr.DataArray(v, [(latname,lat)])
            else:
                if iflon !='no':
                    v = xr.DataArray(v, [(lonname,lon)])
                else:
                    v = None
        levels = None
    if iftime !='no':
        times = v[timename]
    else:
        times = None
    if iflon !='no':
        lon = v[lonname]
    else:
        lon = None
    if iflat !='no':
        lat = v[latname]
    else:
        lat = None
    return v,lon,lat,levels,latlow,lattop,lonleft,lonright,times


"""Small I/O and memory utilities for residual-model inference."""

import gc
import hashlib
import os
from pathlib import Path
import tempfile

import numpy as np


def rounded_hour_times(values, tolerance_ns=1_000_000):
    """Return unique, increasing datetime64[ns] hours, tolerating tiny NC drift."""
    if (
        isinstance(tolerance_ns, (bool, np.bool_))
        or not isinstance(tolerance_ns, (int, np.integer))
        or not 0 <= tolerance_ns < 1_800_000_000_000
    ):
        raise ValueError("tolerance_ns must be an integer from 0 to half an hour")
    raw = np.asarray(values)
    if raw.ndim != 1 or raw.size == 0:
        raise ValueError("time must be a nonempty one-dimensional coordinate")
    if raw.dtype.kind in "biufc":
        raise TypeError("time must contain datetimes, not undecoded numeric values")
    try:
        times = raw.astype("datetime64[ns]")
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError("time cannot be decoded as datetime64[ns]") from exc
    if np.isnat(times).any():
        raise ValueError("time contains NaT")
    nanos = times.view(np.int64)
    hour_ns = 3_600_000_000_000
    remainder = np.remainder(nanos, hour_ns)
    offsets = np.where(remainder <= hour_ns // 2, -remainder, hour_ns - remainder)
    if np.any(np.abs(offsets) > tolerance_ns):
        raise ValueError("time differs from its nearest hour by more than tolerance_ns")
    rounded = nanos + offsets
    if np.any(((offsets > 0) & (rounded < nanos)) | ((offsets < 0) & (rounded > nanos))):
        raise ValueError("rounded time exceeds the datetime64[ns] range")
    if np.any(rounded[1:] <= rounded[:-1]):
        raise ValueError("rounded time must be strictly increasing and unique")
    return rounded.view("datetime64[ns]")


def _wind_coordinate(values, name, length):
    """Validate a finite, real, one-dimensional spatial coordinate."""
    coordinate = np.asarray(values)
    if coordinate.ndim != 1 or coordinate.size != length:
        raise ValueError(f"{name} must have shape ({length},)")
    if coordinate.dtype.kind not in "iuf" or not np.isfinite(coordinate).all():
        raise ValueError(f"{name} must contain finite real numbers")
    if np.any(np.abs(coordinate.astype(np.float64)) > np.finfo(np.float32).max):
        raise ValueError(f"{name} exceeds the float32 range")
    result = coordinate.astype(np.float32)
    if np.unique(result).size != length:
        raise ValueError(f"{name} must remain unique after float32 conversion")
    return result


def _wind_nc_attributes(attrs):
    """Normalize scalar metadata to the string/numeric types accepted by NetCDF."""
    result = {}
    for key, value in (attrs or {}).items():
        if not isinstance(key, str) or not key:
            raise TypeError("NetCDF attribute names must be nonempty strings")
        if value is None:
            continue
        if isinstance(value, (bool, np.bool_)):
            value = int(value)
        elif isinstance(value, (np.integer, np.floating)):
            value = value.item()
        if not isinstance(value, (str, int, float)):
            raise TypeError(f"NetCDF attribute {key!r} must be a scalar string or number")
        if isinstance(value, float) and not np.isfinite(value):
            raise ValueError(f"NetCDF attribute {key!r} must be finite")
        result[key] = value
    return result


def write_wind_nc(
    path, variable, values, times, lat, lon, levels,
    mean=None, std=None, attrs=None, chunk_size=128,
):
    """Atomically write N,L,H,W winds, inverse-normalizing one time chunk at a time.

    ``mean`` and ``std`` must both be omitted or both have shape L,H,W. Metadata
    in ``attrs`` is saved globally; standard field metadata is also attached to
    the wind variable. Supply w units and its saved -0.1*omega convention there.
    No NaN, infinity, masked cell, or float32 overflow is replaced or filled.
    """
    import netCDF4

    if not isinstance(variable, str) or not variable or variable in {
        "time", "level", "latitude", "longitude",
    }:
        raise ValueError("variable must be a nonempty field name, distinct from coordinates")
    if (
        isinstance(chunk_size, (bool, np.bool_))
        or not isinstance(chunk_size, (int, np.integer))
        or chunk_size < 1
    ):
        raise ValueError("chunk_size must be a positive integer")
    data = np.asanyarray(values)
    if data.ndim != 4 or any(size == 0 for size in data.shape):
        raise ValueError("values must have nonempty shape (time, level, latitude, longitude)")
    if data.dtype.kind not in "iuf":
        raise TypeError("values must have a real numeric dtype")
    count, depth, height, width = data.shape
    time_values = rounded_hour_times(times)
    if time_values.size != count:
        raise ValueError("time length does not match values.shape[0]")
    lat_values = _wind_coordinate(lat, "latitude", height)
    lon_values = _wind_coordinate(lon, "longitude", width)
    level_values = _wind_coordinate(levels, "level", depth)
    if (mean is None) != (std is None):
        raise ValueError("mean and std must be supplied together")
    if mean is not None:
        mean_values = np.asarray(mean, dtype=np.float64)
        std_values = np.asarray(std, dtype=np.float64)
        if mean_values.shape != data.shape[1:] or std_values.shape != data.shape[1:]:
            raise ValueError("mean and std must both have shape (level, latitude, longitude)")
        if not np.isfinite(mean_values).all() or not np.isfinite(std_values).all():
            raise ValueError("mean and std must be finite")
        if np.any(std_values <= 0):
            raise ValueError("std must be strictly positive")
    metadata = _wind_nc_attributes(attrs)
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(
        prefix=f".{output_path.name}.", suffix=".tmp", dir=output_path.parent,
    )
    os.close(descriptor)
    try:
        with netCDF4.Dataset(temporary_name, "w", format="NETCDF4") as dataset:
            dimensions = ("time", "level", "latitude", "longitude")
            for name, size in zip(dimensions, data.shape):
                dataset.createDimension(name, size)
            dataset.setncatts({"Conventions": "CF-1.8", **metadata})
            time_var = dataset.createVariable("time", "i8", ("time",), fill_value=False)
            time_var.setncatts({
                "units": "seconds since 1970-01-01 00:00:00",
                "calendar": "standard", "standard_name": "time", "axis": "T",
            })
            time_var[:] = time_values.astype("datetime64[s]").astype(np.int64)
            coordinate_specs = (
                ("level", level_values, {"units": "hPa", "long_name": "pressure level", "positive": "down", "axis": "Z"}),
                ("latitude", lat_values, {"units": "degrees_north", "standard_name": "latitude", "axis": "Y"}),
                ("longitude", lon_values, {"units": "degrees_east", "standard_name": "longitude", "axis": "X"}),
            )
            for name, coordinate, coordinate_attrs in coordinate_specs:
                coordinate_var = dataset.createVariable(name, "f4", (name,), fill_value=False)
                coordinate_var.setncatts(coordinate_attrs)
                coordinate_var[:] = coordinate
            field = dataset.createVariable(
                variable, "f4", dimensions, zlib=True, complevel=1, shuffle=True,
                chunksizes=(min(int(chunk_size), count), 1, height, width),
                fill_value=False,
            )
            if variable.lower().split("_")[-1] in {"u", "v"}:
                field.setncattr("units", "m s-1")
            field_metadata_keys = {
                "units", "long_name", "standard_name", "comment", "vertical_velocity_note",
            }
            field.setncatts({key: value for key, value in metadata.items() if key in field_metadata_keys})
            limit = np.finfo(np.float32).max
            for start in range(0, count, int(chunk_size)):
                stop = min(start + int(chunk_size), count)
                source = data[start:stop]
                if np.ma.isMaskedArray(source) and np.ma.getmaskarray(source).any():
                    raise ValueError(f"masked wind data in time slice {start}:{stop}")
                chunk = np.array(source, dtype=np.float64, copy=True)
                if not np.isfinite(chunk).all():
                    raise ValueError(f"nonfinite wind data in time slice {start}:{stop}")
                if mean is not None:
                    with np.errstate(over="ignore", invalid="ignore"):
                        chunk *= std_values
                        chunk += mean_values
                if not np.isfinite(chunk).all() or np.any(np.abs(chunk) > limit):
                    raise ValueError(f"wind data exceeds finite float32 range in time slice {start}:{stop}")
                field[start:stop] = chunk.astype(np.float32)
        os.replace(temporary_name, output_path)
    except BaseException:
        try:
            os.unlink(temporary_name)
        except FileNotFoundError:
            pass
        raise
    return output_path


def read_wind_array(ds, variable):
    """Return a lazy DataArray with canonical axes and exact hourly timestamps."""
    field = ds[variable]
    aliases = {old: new for old, new in (("lat", "latitude"), ("lon", "longitude")) if old in field.dims}
    if aliases:
        field = field.rename(aliases)
    dimensions = ("time", "level", "latitude", "longitude")
    if field.ndim != 4 or set(field.dims) != set(dimensions):
        raise ValueError(f"{variable} must have exactly the dimensions {dimensions}; found {field.dims}")
    field = field.transpose(*dimensions)
    if any(field.sizes[name] == 0 for name in dimensions):
        raise ValueError(f"{variable} has an empty dimension")
    if field.dtype.kind not in "iuf":
        raise TypeError(f"{variable} must have a real numeric dtype")
    for name in dimensions:
        if name not in field.coords or field.coords[name].dims != (name,):
            raise ValueError(f"{variable} requires a one-dimensional {name} coordinate")
        if field.coords[name].size != field.sizes[name]:
            raise ValueError(f"{variable} has a mismatched {name} coordinate")
    times = rounded_hour_times(field.coords["time"].values)
    for name in dimensions[1:]:
        _wind_coordinate(field.coords[name].values, name, field.sizes[name])
    return field.assign_coords(time=times)


def clear_model_memory(model=None):
    """Clear only this model's ordinary caches, then release unused CUDA memory.

    Registered parameters, buffers, and submodules are preserved, and the model
    is moved to CPU. The caller must remove its own model/optimizer references.
    """
    try:
        import torch
    except ImportError:
        if model is not None:
            raise
        gc.collect()
        return
    if model is not None:
        if not isinstance(model, torch.nn.Module):
            raise TypeError("model must be a torch.nn.Module or None")
        pending = [model]
        visited = set()
        while pending:
            current = pending.pop()
            if id(current) in visited:
                continue
            visited.add(id(current))
            pending.extend(current._modules.values())
            pending = [module for module in pending if module is not None]
            for name, value in list(current.__dict__.items()):
                if name in {"_parameters", "_buffers", "_modules"}:
                    continue
                if name in {"self", "align_residual"} or isinstance(value, (torch.Tensor, np.ndarray)):
                    current.__dict__.pop(name, None)
            del current, value
        model.to("cpu")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def checkpoint_sha256(path):
    """Compute a checkpoint provenance digest without loading it into memory."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


"""Load saved split-specific VMD modes without constructing the full window tensor.

The three contexts remain separate because overlapping timestamps can have different
VMD values. A sample uses the earliest context containing its entire input window.
Mode statistics use the unique input hours in retained training windows.
"""
from pathlib import Path
import warnings

import numpy as np
import xarray as xr


_VMD_SPLITS = ("train", "valid", "test")


def _positive_integer(value, name):
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value < 1:
        raise ValueError(f"{name} must be a positive integer")
    return int(value)


def _window_indices(indices, length):
    indices = np.asarray(indices)
    if indices.ndim != 1 or (indices.size and indices.dtype.kind not in "iu"):
        raise IndexError("window indices must be a one-dimensional integer sequence")
    if indices.dtype.kind == "u" and np.any(indices >= length):
        raise IndexError("window index out of range")
    indices = indices.astype(np.int64, copy=False)
    indices = np.where(indices < 0, indices + length, indices)
    if np.any((indices < 0) | (indices >= length)):
        raise IndexError("window index out of range")
    return indices


class WindowSelection:
    """Lazy sample selection; only scalar access or np.asarray materializes data."""

    ndim = 5
    dtype = np.dtype(np.float32)

    def __init__(self, provider, indices=None):
        self.provider = provider
        self.indices = (np.arange(provider.shape[0], dtype=np.int64) if indices is None
                        else _window_indices(indices, provider.shape[0]))
        self.shape = (len(self.indices),) + tuple(provider.shape[1:])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        selected = self.indices[index]
        if np.ndim(selected) == 0:
            return self.provider.take(np.asarray([selected], dtype=np.int64))[0]
        if np.ndim(selected) != 1:
            raise IndexError("WindowSelection supports indexing its sample axis only")
        return WindowSelection(self.provider, selected)

    def __array__(self, dtype=None, copy=None):
        if copy is False:
            raise ValueError("materializing a WindowSelection requires a copy")
        result = self.provider.take(self.indices)
        return result if dtype is None else result.astype(dtype, copy=False)


class ArrayWindowProvider:
    """Build N,T,C,H,W minibatches from an N,H,W,C input array on demand."""

    ndim = 5

    def __init__(self, vx, timestep):
        self.vx = np.asarray(vx)
        self.timestep = _positive_integer(timestep, "timestep")
        if self.vx.ndim != 4 or any(size < 1 for size in self.vx.shape):
            raise ValueError("vx must be a nonempty (time, latitude, longitude, channel) array")
        if len(self.vx) < self.timestep:
            raise ValueError("vx contains fewer time steps than timestep")
        self.shape = (len(self.vx) - self.timestep + 1, self.timestep,
                      self.vx.shape[3], self.vx.shape[1], self.vx.shape[2])

    def take(self, indices):
        indices = _window_indices(indices, self.shape[0])
        positions = indices[:, None] + np.arange(self.timestep)[None, :]
        result = np.ascontiguousarray(self.vx[positions].transpose(0, 1, 4, 2, 3), dtype=np.float32)
        return np.nan_to_num(result, copy=False)


class VMDWindowProvider:
    """Combine raw predictors with modes from one context per complete window."""

    ndim = 5

    def __init__(self, vx_raw, timestep, contexts, context_ranges, context_codes):
        self.raw = ArrayWindowProvider(vx_raw, timestep)
        self.timestep = self.raw.timestep
        self.contexts = contexts
        self.context_ranges = context_ranges
        self.context_codes = np.asarray(context_codes, dtype=np.int8)
        mode_count = contexts[0].shape[-1]
        self.shape = (self.raw.shape[0], self.timestep, self.raw.shape[2] + mode_count,
                      self.raw.shape[3], self.raw.shape[4])

    def take(self, indices):
        indices = _window_indices(indices, self.shape[0])
        result = np.empty((len(indices),) + self.shape[1:], dtype=np.float32)
        raw_channels = self.raw.shape[2]
        result[:, :, :raw_channels] = self.raw.take(indices)
        selected_codes = self.context_codes[indices]
        for code, modes in enumerate(self.contexts):
            rows = np.flatnonzero(selected_codes == code)
            local_starts = indices[rows] - self.context_ranges[code][0]
            positions = local_starts[:, None] + np.arange(self.timestep)[None, :]
            result[rows, :, raw_channels:] = modes[positions].transpose(0, 1, 4, 2, 3)
        return np.nan_to_num(result, copy=False)


def _vmd_hour_times(values, label):
    """Accept up to 1 ms of NetCDF drift, then require unique consecutive hours."""
    values = np.asarray(values)
    if values.ndim != 1 or not values.size or values.dtype.kind in "biufc":
        raise ValueError(f"{label}: time must be a nonempty decoded datetime coordinate")
    try:
        times = values.astype("datetime64[ns]")
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError(f"{label}: time cannot be decoded as datetime64[ns]") from exc
    if np.isnat(times).any():
        raise ValueError(f"{label}: time contains NaT")
    nanos = times.view(np.int64)
    hour_ns = 3_600_000_000_000
    remainder = np.remainder(nanos, hour_ns)
    offsets = np.where(remainder <= hour_ns // 2, -remainder, hour_ns - remainder)
    if np.any(np.abs(offsets) > 1_000_000):
        raise ValueError(f"{label}: time differs from its nearest hour by more than 1 ms")
    rounded = nanos + offsets
    if np.any(((offsets > 0) & (rounded < nanos)) | ((offsets < 0) & (rounded > nanos))):
        raise ValueError(f"{label}: rounded time exceeds datetime64[ns] range")
    if np.any(rounded[1:] <= rounded[:-1]):
        raise ValueError(f"{label}: time must be increasing and unique")
    if np.any(np.diff(rounded) != hour_ns):
        raise ValueError(f"{label}: time must be a consecutive hourly grid")
    return rounded.view("datetime64[ns]")


def _spatial_coordinate(values, name):
    values = np.asarray(values)
    if values.ndim != 1 or not values.size or values.dtype.kind not in "iuf":
        raise ValueError(f"{name} must be a nonempty real one-dimensional coordinate")
    values = values.astype(np.float64)
    if not np.isfinite(values).all() or len(np.unique(values)) != len(values):
        raise ValueError(f"{name} must contain finite unique coordinates")
    return values


def _coordinate_order(source, target, name, path):
    source = _spatial_coordinate(source, name)
    if len(source) != len(target):
        raise ValueError(f"{path}: {name} size differs from the raw predictor grid")
    matches = np.isclose(target[:, None], source[None, :], rtol=0, atol=1e-6)
    if not (np.all(matches.sum(axis=1) == 1) and np.all(matches.sum(axis=0) == 1)):
        raise ValueError(f"{path}: {name} coordinates do not match the raw predictor grid")
    return matches.argmax(axis=1)


def _axis_coordinate(data, aliases, label, path):
    for name in aliases:
        if name in data.coords:
            coordinate = data.coords[name]
            if coordinate.ndim == 1 and coordinate.dims[0] in data.dims:
                return name, coordinate.dims[0], coordinate.values
    raise ValueError(f"{path}: missing one-dimensional {label} coordinate ({', '.join(aliases)})")


def _json_value(value):
    if isinstance(value, np.ndarray):
        return _json_value(value.tolist())
    if isinstance(value, (list, tuple)):
        return [_json_value(item) for item in value]
    if isinstance(value, dict):
        return {str(key): _json_value(item) for key, item in value.items()}
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return str(value)
    if value is None or isinstance(value, (str, bool, int, float)):
        return value
    return str(value)


def _time_range(times):
    return [str(times[0]), str(times[-1])] if len(times) else None


def load_vmd_window_provider(vx_raw, raw_times, forecast_hours, timestep, lat, lon,
                             specs, *, train_stop, vmd_dir=None, chunk_size=128):
    """Validate saved modes, fit mode statistics on unique hours in retained training windows, and expose windows.

    ``vx_raw`` contains raw SP/T2M/D2M inputs (standardized in place after splitting) and has length
    ``len(raw_times) - forecast_hours``. ``specs`` is an ordered sequence of
    ``{key, stem, K}`` dictionaries. All split files are checked before mode arrays
    are allocated. Returned mean/std arrays have shape (latitude, longitude, mode).
    ``np.asarray(WindowSelection(provider)[start:stop])`` creates one minibatch.
    """
    forecast_hours = _positive_integer(forecast_hours, "forecast_hours")
    timestep = _positive_integer(timestep, "timestep")
    chunk_size = _positive_integer(chunk_size, "chunk_size")
    vx_raw = np.asarray(vx_raw)
    raw_provider = ArrayWindowProvider(vx_raw, timestep)
    if vx_raw.shape[-1] != 3:
        raise ValueError("vx_raw must contain exactly the three raw channels: sp, st, sd")
    all_times = _vmd_hour_times(raw_times, "raw_times")
    if len(vx_raw) != len(all_times) - forecast_hours:
        raise ValueError("vx_raw length must equal len(raw_times) - forecast_hours")
    input_times = all_times[:len(vx_raw)]
    lat = _spatial_coordinate(lat, "latitude")
    lon = _spatial_coordinate(lon, "longitude")
    if (len(lat), len(lon)) != vx_raw.shape[1:3]:
        raise ValueError("raw predictor shape does not match latitude/longitude coordinates")
    if vmd_dir is None:
        vmd_dir = globals().get("VMD_DIR")
    if vmd_dir is None:
        raise ValueError("vmd_dir is required (or define VMD_DIR)")
    vmd_dir = Path(vmd_dir)
    specs = [{"key": str(spec["key"]), "stem": str(spec["stem"]),
              "K": _positive_integer(spec["K"], "K")} for spec in specs]
    if not specs or len({spec["key"] for spec in specs}) != len(specs):
        raise ValueError("specs must contain distinct variable keys")
    channels = [(spec, k, f'{spec["key"]}_imf_{k}')
                for spec in specs for k in range(1, spec["K"] + 1)]
    files = [(code, split, channel, spec, k, variable,
              vmd_dir / f'{spec["stem"]}_imf_{k}_{split}.nc')
             for code, split in enumerate(_VMD_SPLITS)
             for channel, (spec, k, variable) in enumerate(channels)]
    missing = [str(entry[-1]) for entry in files if not entry[-1].is_file()]
    if missing:
        raise FileNotFoundError("Missing VMD files:\n" + "\n".join(missing))

    # Read headers/coordinates from every file before allocating the mode arrays.
    headers, sources = [], []
    context_times = [None] * len(_VMD_SPLITS)
    context_ranges = [None] * len(_VMD_SPLITS)
    for code, split, channel, spec, k, variable, path in files:
        with xr.open_dataset(path) as dataset:
            if variable not in dataset.data_vars:
                raise ValueError(f"{path}: missing data variable {variable}")
            data = dataset[variable]
            for attrs in (dataset.attrs, data.attrs):
                if "vmd_write_complete" in attrs:
                    try:
                        complete = float(attrs["vmd_write_complete"])
                    except (TypeError, ValueError, OverflowError) as exc:
                        raise ValueError(f"{path}: invalid vmd_write_complete metadata") from exc
                    if complete != 1:
                        raise ValueError(f"{path}: vmd_write_complete must equal 1")
                for attribute in ("vmd_K", "selected_K"):
                    if attribute in attrs:
                        try:
                            reported_k = float(attrs[attribute])
                        except (TypeError, ValueError, OverflowError) as exc:
                            raise ValueError(f"{path}: invalid {attribute} metadata") from exc
                        if reported_k != spec["K"]:
                            raise ValueError(f"{path}: {attribute}={attrs[attribute]} differs from expected K={spec['K']}")
            tname, tdim, times = _axis_coordinate(data, ("time", "valid_time"), "time", path)
            yname, ydim, yvalues = _axis_coordinate(data, ("lat", "latitude"), "latitude", path)
            xname, xdim, xvalues = _axis_coordinate(data, ("lon", "longitude"), "longitude", path)
            if data.ndim != 3 or len({tdim, ydim, xdim}) != 3:
                raise ValueError(f"{path}: {variable} must have exactly time/latitude/longitude dimensions")
            times = _vmd_hour_times(times, str(path))
            lat_order = _coordinate_order(yvalues, lat, "latitude", path)
            lon_order = _coordinate_order(xvalues, lon, "longitude", path)
            first = int(np.searchsorted(times, input_times[0], side="left"))
            last = int(np.searchsorted(times, input_times[-1], side="right"))
            selected_times = times[first:last]
            if len(selected_times):
                global_first = int(np.searchsorted(input_times, selected_times[0]))
                global_last = global_first + len(selected_times)
                if not np.array_equal(selected_times, input_times[global_first:global_last]):
                    raise ValueError(f"{path}: selected VMD times do not align with raw input times")
            else:
                global_first = global_last = 0
            if context_times[code] is None:
                context_times[code] = selected_times
                context_ranges[code] = (global_first, global_last)
            elif not np.array_equal(context_times[code], selected_times):
                raise ValueError(f"{path}: all modes in {split} must have identical selected timestamps")
            sources.append({
                "path": str(path), "split": split, "variable": variable, "K": spec["K"],
                "source_shape": list(data.shape), "source_dimensions": list(data.dims),
                "selected_shape": [len(selected_times), len(lat), len(lon)],
                "source_time_range": _time_range(times), "selected_time_range": _time_range(selected_times),
                "coordinate_names": {"time": tname, "latitude": yname, "longitude": xname},
                "latitude_order": lat_order.tolist(), "longitude_order": lon_order.tolist(),
                "dataset_attrs": _json_value(dict(dataset.attrs)), "variable_attrs": _json_value(dict(data.attrs)),
            })
            headers.append((code, channel, variable, path, tdim, ydim, xdim,
                            first, last, lat_order, lon_order))

    # Timeline ownership is for statistics only; windows use a single split below.
    timeline_owner = np.full(len(input_times), -1, dtype=np.int8)
    context_codes = np.full(raw_provider.shape[0], -1, dtype=np.int8)
    sample_starts = np.arange(raw_provider.shape[0], dtype=np.int64)
    for code, (start, stop) in enumerate(context_ranges):
        timeline_owner[start:stop] = np.where(timeline_owner[start:stop] < 0, code,
                                               timeline_owner[start:stop])
        contained = (sample_starts >= start) & (sample_starts + timestep <= stop)
        context_codes[(context_codes < 0) & contained] = code
    if np.any(timeline_owner < 0):
        missing_time = input_times[np.flatnonzero(timeline_owner < 0)[0]]
        raise ValueError(f"VMD contexts do not cover input hour {missing_time}")
    if np.any(context_codes < 0):
        missing_start = int(np.flatnonzero(context_codes < 0)[0])
        raise ValueError("No single VMD context contains the complete input window "
                         f"starting at {input_times[missing_start]}; cannot mix split contexts")

    mode_count = len(channels)
    contexts = [np.empty((stop - start, len(lat), len(lon), mode_count), dtype=np.float32)
                for start, stop in context_ranges]
    for code, channel, variable, path, tdim, ydim, xdim, first, last, lat_order, lon_order in headers:
        with xr.open_dataset(path) as dataset:
            data = dataset[variable].transpose(tdim, ydim, xdim)
            for offset in range(0, last - first, chunk_size):
                end = min(offset + chunk_size, last - first)
                chunk = np.asarray(data.isel({tdim: slice(first + offset, first + end)}).values,
                                   dtype=np.float32)
                contexts[code][offset:end, :, :, channel] = chunk[:, lat_order][:, :, lon_order]

    means = np.empty((len(lat), len(lon), mode_count), dtype=np.float32)
    stds = np.empty_like(means)
    normalization = []
    training_starts = sample_starts[(sample_starts < train_stop) & (context_codes == 0)]
    training_hours = np.unique(training_starts[:, None] + np.arange(timestep))
    local_hours = training_hours - context_ranges[0][0]
    for channel, (_, _, variable) in enumerate(channels):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            train_mode = contexts[0][local_hours, :, :, channel]
            mean = np.nanmean(train_mode, axis=0)
            raw_std = np.nanstd(train_mode, axis=0)
            del train_mode
        if not np.isfinite(mean).all() or not np.isfinite(raw_std).all():
            raise ValueError(f"{variable}: non-finite training normalization statistics")
        constant = raw_std <= 1e-12
        std = np.where(constant, np.float32(1), raw_std)
        means[..., channel], stds[..., channel] = mean, std
        normalization.append({"channel": variable, "constant_cell_count": int(constant.sum()),
                              "raw_std_min": float(raw_std.min()), "raw_std_max": float(raw_std.max())})
    for modes in contexts:
        for start in range(0, len(modes), chunk_size):
            chunk = modes[start:start + chunk_size]
            np.subtract(chunk, means, out=chunk)
            np.divide(chunk, stds, out=chunk)

    provider = VMDWindowProvider(vx_raw, timestep, contexts, context_ranges, context_codes)
    boundaries = []
    changes = np.r_[0, np.flatnonzero(np.diff(context_codes)) + 1, len(context_codes)]
    for first, stop in zip(changes[:-1], changes[1:]):
        boundaries.append({"split": _VMD_SPLITS[int(context_codes[first])],
                           "sample_start": int(first), "sample_stop_exclusive": int(stop),
                           "input_start_time_range": _time_range(input_times[first:stop]),
                           "input_end_time_range": _time_range(input_times[first + timestep - 1:stop + timestep - 1])})
    metadata = {
        "specs": specs, "forecast_hours": forecast_hours, "timestep": timestep,
        "channel_order": ["sp", "st", "sd"] + [item[2] for item in channels],
        "provider_shape": list(provider.shape), "input_time_range": _time_range(input_times),
        "latitude": lat.tolist(), "longitude": lon.tolist(), "sources": sources,
        "window_ownership": "earliest train/valid/test context containing the complete input window",
        "normalization_scope": "unique hours in retained training windows, per grid cell and mode, ddof=0",
        "normalization_training_hour_count": len(training_hours),
        "timeline_owned_hour_counts": {split: int(np.count_nonzero(timeline_owner == code))
                                            for code, split in enumerate(_VMD_SPLITS)},
        "normalization": normalization,
        "selected_window_counts": {split: int(np.count_nonzero(context_codes == code))
                                   for code, split in enumerate(_VMD_SPLITS)},
        "context_boundaries": boundaries,
    }
    return provider, means, stds, metadata


In [ ]:
#用于生成DA-Transunet网络pytorch版本
def Auto_DA_TransUnet_pytorch(vy,vx,timestep,downnum,covnum,test_size=0.2,valid_size=0.1,k_fold=None,task_mode='regression',if_best_mode='no',modelpath=None,ifrandom_split='yes',if_bottom_attention_only='no',if_resnet='no',layer_time=2,baselayer=64,cov_kernelsize=3,cov_strides=1,pool_method='maxpooling',pool_kernel_size=2,pool_strides=2,ifnormalization='no',normalization_method=None,da_attention_order='space_first',trans_num=2,num_heads=2,dim_feedforward=256,trans_dropout_rate=0.0,activate='tanh',nlp_activate='relu',if_last_act='no',if_print_model='yes',loss_function='default',optimizer='SGD',metrics='default',learning_rate=0.01,epochs=2000,batch_size=20,if_early_stopping=None,ifheatmap='no',ifweight='yes',ifmute='no',ifsave='no',savepath=None,device='cpu',window_provider=None,predefined_split=None):
    from torch.nn import Module,BatchNorm1d,BatchNorm2d,LayerNorm,Conv2d,MaxPool2d,AvgPool2d,Dropout,LeakyReLU,ReLU,PReLU,Sigmoid,Tanh,ELU,Softmax,Linear,Flatten,ZeroPad2d,Upsample,TransformerEncoderLayer,TransformerEncoder
    from torch.optim import Adam,NAdam,SGD
    import torch
    from torch import nn
    import torchmetrics
    import math
    from sklearn.model_selection import train_test_split
    from sklearn.model_selection import KFold
    import numpy as np
    from scipy.stats import pearsonr
    import os
    from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score
    import sklearn
    import copy
    import shap
    import datetime
    import warnings
    from tqdm import tqdm
    warnings.filterwarnings('ignore')
    
    if device=='gpu':
        devices=torch.device('cuda:1')
    elif str(device).startswith('cuda'):
        devices=torch.device(device)
    else:
        devices=torch.device('cpu')
    if task_mode=='binary_classify' or task_mode=='multi_classify':
        def get_class_weights_from_labels(vy, device=None, ignore_index=None, num_classes=None):
            y = torch.as_tensor(vy, device=device)
        
            # 如果你标签存成 (...,1)，内部 squeeze 掉这个 1（不改原 vy）
            if y.ndim >= 1 and y.shape[-1] == 1:
                y = y.squeeze(-1)
        
            y = y.reshape(-1)
        
            if ignore_index is not None:
                y = y[y != ignore_index]
        
            if y.is_floating_point():
                y = y[~torch.isnan(y)]
        
            y = y.long()
        
            if num_classes is None:
                num_classes = int(y.max().item()) + 1
        
            cnt = torch.bincount(y, minlength=num_classes).float()
            w = cnt.sum() / (num_classes * cnt.clamp_min(1.0))  # N/(C*count_c)
            return w  # shape: (C,)
        class_weights = get_class_weights_from_labels(vy, device=devices)
    if task_mode=='regression':
        if type(loss_function) is not str:
            loss=loss_function
        else:
            if loss_function=='default' or loss_function=='MSELoss':
                loss=torch.nn.MSELoss()
            elif loss_function=='L1Loss':
                loss=torch.nn.L1Loss
            elif loss_function=='PoissonNLLLoss':
                loss=torch.nn.PoissonNLLLoss()
            elif loss_function=='GaussianNLLLoss':
                loss=torch.nn.GaussianNLLLoss()
            elif loss_function=='KLDivLoss':
                loss=torch.nn.KLDivLoss()
            elif loss_function=='HuberLoss':
                loss=torch.nn.HuberLoss()
            elif loss_function=='SmoothL1Loss':
                loss=torch.nn.SmoothL1Loss()
            elif loss_function=='Pearsonr':
                class loss_pearsonr(nn.Module):
                    def __init__(self):
                        super().__init__()
    
                    def forward(self, y, x):
                        y_true_mean=torch.nanmean(y,dim=0,keepdim=True)
                        y_pred_mean=torch.nanmean(x,dim=0,keepdim=True)
                        cov=torch.nansum((y-y_true_mean)*(x-y_pred_mean),dim=0,keepdim=True)
                        y_true_v=torch.nansum(torch.square((y-y_true_mean)),dim=0,keepdim=True)
                        y_pred_v=torch.nansum(torch.square((x-y_pred_mean)),dim=0,keepdim=True)
                        y_true_v=torch.sqrt(y_true_v)
                        y_pred_v=torch.sqrt(y_pred_v)
                        pearson=cov/(y_true_v*y_pred_v)
                        return (1-pearson)**1.5
                loss=loss_pearsonr()
        if type(metrics) is not str:
            metric=metrics
        else:
            if metrics=='default' or metrics=='MSELoss':
                metric=torch.nn.MSELoss()
            elif metrics=='L1Loss':
                metric=torch.nn.L1Loss
            elif metrics=='PoissonNLLLoss':
                metric=torch.nn.PoissonNLLLoss()
            elif metrics=='GaussianNLLLoss':
                metric=torch.nn.GaussianNLLLoss()
            elif metrics=='KLDivLoss':
                metric=torch.nn.KLDivLoss()
            elif metrics=='HuberLoss':
                metric=torch.nn.HuberLoss()
            elif metrics=='SmoothL1Loss':
                metric=torch.nn.SmoothL1Loss()
            elif metrics=='Pearsonr':
                class metric_pearsonr(nn.Module):
                    def __init__(self):
                        super().__init__()
    
                    def forward(self, y, x):
                        y_true_mean=torch.nanmean(y,dim=0,keepdim=True)
                        y_pred_mean=torch.nanmean(x,dim=0,keepdim=True)
                        cov=torch.nansum((y-y_true_mean)*(x-y_pred_mean),dim=0,keepdim=True)
                        y_true_v=torch.nansum(torch.square((y-y_true_mean)),dim=0,keepdim=True)
                        y_pred_v=torch.nansum(torch.square((x-y_pred_mean)),dim=0,keepdim=True)
                        y_true_v=torch.sqrt(y_true_v)
                        y_pred_v=torch.sqrt(y_pred_v)
                        pearson=cov/(y_true_v*y_pred_v)
                        return (1-pearson)**1.5
                metric=metric_pearsonr()
    elif task_mode=='binary_classify':
        if type(loss_function) is not str:
            loss=loss_function
        else:
            if loss_function=='default' or loss_function=='BCELoss':
                w0, w1 = class_weights[0], class_weights[1]
                weight_map = torch.where(y_batch > 0.5, w1, w0)
                loss=torch.nn.BCELoss(weight=weight_map)
            elif loss_function=='BCEWithLogitsLoss':
                pos_w = (class_weights[1] / class_weights[0]).to(devices).view(1)
                loss=torch.nn.BCEWithLogitsLoss(pos_weight=pos_w)
            elif loss_function=='SoftMarginLoss':
                loss=torch.nn.SoftMarginLoss()
            elif loss_function=='MultiLabelSoftMarginLoss':
                loss=torch.nn.MultiLabelSoftMarginLoss()
        if type(metrics) is not str:
            metric=metrics
        else:
            if metrics=='default' or metrics=='f1':
                metric=torchmetrics.F1Score(task="binary").to(devices)
            elif metrics=='accuracy':
                metric=torchmetrics.Accuracy(task="binary").to(devices)
            elif metrics=='precision':
                metric=torchmetrics.Precision(task="binary").to(devices)
            elif metrics=='recall':
                metric=torchmetrics.Recall(task="binary").to(devices)
            elif metrics=='BCELoss':
                metric=torch.nn.BCELoss()
            elif metrics=='BCEWithLogitsLoss':
                metric=torch.nn.BCEWithLogitsLoss()
            elif metrics=='SoftMarginLoss':
                metric=torch.nn.SoftMarginLoss()
            elif metrics=='MultiLabelSoftMarginLoss':
                metric=torch.nn.MultiLabelSoftMarginLoss()
    elif task_mode=='multi_classify':
        if type(loss_function) is not str:
            loss=loss_function
        else:
            if loss_function=='default' or loss_function=='CrossEntropyLoss':
                loss=torch.nn.CrossEntropyLoss(weight=class_weights)
            elif loss_function=='NLLLoss':
                loss=torch.nn.NLLLoss(weight=class_weights)
            elif loss_function=='TripletMarginLoss':
                loss=torch.nn.TripletMarginLoss()
            elif loss_function=='KLDivergence':
                loss=torch.nn.KLDivergence()
            elif loss_function=='HingeEmbeddingLoss':
                loss=torch.nn.HingeEmbeddingLoss()
            elif loss_function=='MultiLabelMarginLoss':
                loss=torch.nn.MultiLabelMarginLoss()
            elif loss_function=='TripletMarginWithDistanceLoss':
                loss=torch.nn.TripletMarginWithDistanceLoss()
        if type(metrics) is not str:
            metric=metrics
        else:
            if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                metric=torchmetrics.Accuracy(task="multiclass",num_classes = int(np.max(vy))+1, average="macro").to(devices)
            elif metrics=='recall' :
                metric=torchmetrics.Recall(task="multiclass", num_classes=int(np.max(vy))+1, average="macro").to(devices)
            elif metrics=='precision' :
                metric=torchmetrics.Precision(task="multiclass", num_classes=int(np.max(vy))+1, average="macro").to(devices)
            elif metrics=='f1' :
                metric=torchmetrics.F1Score(task="multiclass", num_classes=int(np.max(vy))+1, average="macro").to(devices)
            elif metrics=='CrossEntropyLoss':
                metric=torch.nn.CrossEntropyLoss()
            elif metrics=='NLLLoss':
                metric=torch.nn.NLLLoss()
            elif metrics=='TripletMarginLoss':
                metric=torch.nn.TripletMarginLoss()
            elif metrics=='KLDivergence':
                metric=torch.nn.KLDivergence()
            elif metrics=='HingeEmbeddingLoss':
                metric=torch.nn.HingeEmbeddingLoss()
            elif metrics=='MultiLabelMarginLoss':
                metric=torch.nn.MultiLabelMarginLoss()
            elif metrics=='TripletMarginWithDistanceLoss':
                metric=torch.nn.TripletMarginWithDistanceLoss()
    weights=0
    heatmap = 0
    model=0
    predicty=None
    testy=None
    r=None
    p=None
    class ShapWrapper(Module):
        def __init__(self, original_model, x_sample_shape, output_channel_idx=None):
            super().__init__()
            self.original_model = original_model
            self.x_sample_shape = x_sample_shape
            self.x_flat_size = np.prod(x_sample_shape)
            self.output_channel_idx = output_channel_idx
        
        def forward(self, x_flat_tensor):
            if x_flat_tensor.dim() == 1:
                x_flat_tensor = x_flat_tensor.unsqueeze(0)
            
            x_reshaped = x_flat_tensor.reshape(-1, *self.x_sample_shape)
            
            model_output = self.original_model(x_reshaped)
            
            if model_output.is_floating_point() is False:
                model_output = model_output.float()
    
            if self.output_channel_idx is not None and model_output.ndim == 4:
                return torch.mean(model_output[:, self.output_channel_idx, :, :].view(model_output.size(0), -1), dim=1).unsqueeze(1)
            elif model_output.ndim == 4:
                return torch.mean(model_output.view(model_output.size(0), -1), dim=1).unsqueeze(1)
            else:
                return torch.mean(model_output.view(model_output.size(0), -1), dim=1).unsqueeze(1)
    class EarlyStopping:
        def __init__(self, patience, delta=0):
            self.patience = patience
            self.counter = 0
            self.best_score = None
            self.early_stop = False
            self.val_loss_min = np.inf
            self.delta = delta
            self.best_model_state_dict = None
    
        def __call__(self, val_loss, model):
            # 1. NaN 保护：一旦验证 loss 是 NaN，就停止训练，后面外层用 load_best_checkpoint 恢复
            if np.isnan(val_loss):
                print("Validation loss is NaN. Stopping early and will restore best model.")
                self.early_stop = True
                return
    
            score = -val_loss
    
            # 2. 正常 early stopping 逻辑
            if self.best_score is None:
                self.best_score = score
                self.save_checkpoint(val_loss, model)
            elif score < self.best_score + self.delta:
                # 没有提升
                self.counter += 1
                if self.counter >= self.patience:
                    self.early_stop = True
            else:
                # 有提升，更新 best
                self.best_score = score
                self.save_checkpoint(val_loss, model)
                self.counter = 0
    
        def save_checkpoint(self, val_loss, model):
            # 这里仍然是 clone 当前最优模型
            self.best_model_state_dict = {k: v.clone() for k, v in model.state_dict().items()}
            self.val_loss_min = val_loss
    
        def load_best_checkpoint(self, model):
            # 新增：一行恢复 best 模型
            if self.best_model_state_dict is not None:
                model.load_state_dict(self.best_model_state_dict)
            else:
                print("Warning: load_best_checkpoint called but no best_model_state_dict was saved.")
    if vy.ndim==3:
        vy=vy.reshape(vy.shape[0],vy.shape[1],vy.shape[2],1)
    vy=vy.transpose(0,3,1,2)
    vys=vy[timestep-1:,:,:,:]
    # 可选分段输入接口：只改变取数方式，保留网络类、exec风格和优化过程。
    # VMD分段交界的同一时次可能有不同IMF，必须为完整历史窗口选同一分段。
    if window_provider is None:
        vxs=np.zeros((vx.shape[0]-timestep+1,timestep,vx.shape[1],vx.shape[2],vx.shape[3]),dtype=np.float32)
        for i in range(vx.shape[0]-timestep+1):
            vxs[i,:,:,:,:]=vx[i:i+timestep,:,:,:]
        vxs=vxs.transpose(0,1,4,2,3)
    else:
        if k_fold is not None or ifrandom_split not in ('no','all_test') or ifheatmap!='no' or ifweight!='no':
            raise ValueError('分段窗口接口用于当前顺序切分、无K折/SHAP的训练及all_test推理。')
        vxs=WindowSelection(window_provider)
        if vxs.shape[0]!=len(vys) or vxs.shape[1]!=timestep or vxs.shape[-2:]!=vy.shape[-2:]:
            raise ValueError('window_provider 与目标数量、历史窗口或空间尺寸不一致。')
    _explicit_train_count=None
    if predefined_split is not None and ifrandom_split=='no':
        _ids={name: np.asarray(predefined_split[name],dtype=np.int64) for name in ('train','valid','test')}
        for name,idx in _ids.items():
            if idx.ndim!=1 or len(idx)==0 or np.any(np.diff(idx)<=0) or idx[0]<0 or idx[-1]>=len(vys):
                raise ValueError(f'{name}: 预设样本索引不合法。')
        if _ids['train'][-1]>=_ids['valid'][0] or _ids['valid'][-1]>=_ids['test'][0]:
            raise ValueError('预设train/valid/test不是互不重叠的顺序切分。')
        if valid_size is None or k_fold is not None:
            raise ValueError('预设切分需要独立验证集，且不使用K折。')
        _trainval=np.concatenate((_ids['train'],_ids['valid']))
        _explicit_train_count=len(_ids['train'])
        trainy,testy=vys[_trainval],vys[_ids['test']]
        trainx,testx=vxs[_trainval],vxs[_ids['test']]
    elif ifrandom_split=='yes':
        trainy,testy,trainx,testx = train_test_split(vys,vxs,test_size=test_size,random_state=25)
    elif ifrandom_split=='no':
        index=int((1-test_size)*vy.shape[0])
        trainy=vys[:index]
        testy=vys[index:]
        trainx=vxs[:index]
        testx=vxs[index:]
    elif ifrandom_split=='all_train' or ifrandom_split=='all_test' or ifrandom_split=='just_model':
        trainy=vys
        testy=vys
        trainx=vxs
        testx=vxs
    class TimeDistributed(nn.Module):
        def __init__(self, module, batch_first=False):
            super(TimeDistributed, self).__init__()
            self.module = module
            self.batch_first = batch_first
        def forward(self, x): 
            if self.batch_first:
                batch_size, seq_len = x.shape[0], x.shape[1]
            else:
                seq_len, batch_size = x.shape[0], x.shape[1]
            x_reshaped = x.view(batch_size * seq_len, *x.shape[2:])
            y = self.module(x_reshaped)
            if self.batch_first:
                y = y.view(batch_size, seq_len, *y.shape[1:])
            else:
                y = y.view(seq_len, batch_size, *y.shape[1:])

            return y
    
    class Model(nn.Module):
        def __init__(self,trainx,trainy,timestep,downnum,covnum,layer_time,baselayer,cov_kernelsize,cov_strides,pool_method,pool_kernel_size,pool_strides,ifnormalization,normalization_method,trans_num,num_heads,dim_feedforward,trans_dropout_rate,activate,nlp_activate,da_attention_order,if_bottom_attention_only,if_resnet):
            super(Model,self).__init__()
            exec('from torch.nn import Module,BatchNorm1d,BatchNorm2d,LayerNorm,Conv2d,MaxPool2d,AvgPool2d,Dropout,LeakyReLU,ReLU,PReLU,Sigmoid,Tanh,ELU,Softmax,Linear,Flatten,ZeroPad2d,Upsample,TransformerEncoderLayer,TransformerEncoder', globals(), self.__dict__)
            exec('import numpy as np', globals(), self.__dict__)
            exec('import math', globals(), self.__dict__)
            exec('import torch', globals(), self.__dict__)
            exec('from torch import nn', globals(), self.__dict__)
            self.timestep=timestep
            self.downnum=downnum
            self.covnum=covnum
            self.layer_time=layer_time
            self.baselayer=baselayer
            self.cov_kernelsize=cov_kernelsize
            self.cov_strides=cov_strides
            self.pool_method=pool_method
            self.pool_kernel_size=pool_kernel_size
            self.pool_strides=pool_strides
            self.ifnormalization=ifnormalization
            self.normalization_method=normalization_method
            self.trans_num=trans_num
            self.num_heads=num_heads
            self.dim_feedforward=dim_feedforward
            self.trans_dropout_rate=trans_dropout_rate
            self.activate=activate
            self.nlp_activate=nlp_activate
            self.da_attention_order = da_attention_order
            if self.da_attention_order not in ['time_first', 'space_first']:
                raise ValueError("da_attention_order must be 'time_first' or 'space_first'")
            self.if_bottom_attention_only = if_bottom_attention_only
            self.if_resnet = if_resnet
            
            if self.if_bottom_attention_only not in ['yes', 'no']:
                raise ValueError("if_bottom_attention_only must be 'yes' or 'no'")
            
            if self.if_resnet not in ['yes', 'no']:
                raise ValueError("if_resnet must be 'yes' or 'no'")
            self.trainx=trainx
            self.trainy=trainy
            self.hight0=self.trainx.shape[3]
            self.weight0=self.trainx.shape[4]
            self.upsize=pool_strides
            self.__dict__['self']=self
            self.__dict__['TimeDistributed']=TimeDistributed
            self.k=0
            self.total_covnum=0
            def align_residual(self, shortcut, target):
                """
                shortcut / target:
                    encoder: (B, T, C, H, W)
                    decoder: (B, C, H, W)
                将 shortcut 的 H/W 对齐到 target。
                """
                sh, sw = shortcut.shape[-2], shortcut.shape[-1]
                th, tw = target.shape[-2], target.shape[-1]
    
                # crop height
                if sh > th:
                    dh = sh - th
                    shortcut = shortcut[..., dh // 2: dh // 2 + th, :]
                # crop width
                if sw > tw:
                    dw = sw - tw
                    shortcut = shortcut[..., :, dw // 2: dw // 2 + tw]
    
                sh, sw = shortcut.shape[-2], shortcut.shape[-1]
    
                # pad height / width
                pad_h = th - sh
                pad_w = tw - sw
    
                if pad_h > 0 or pad_w > 0:
                    pad_top = pad_h // 2
                    pad_bottom = pad_h - pad_top
                    pad_left = pad_w // 2
                    pad_right = pad_w - pad_left
                    shortcut = torch.nn.functional.pad(
                        shortcut,
                        (pad_left, pad_right, pad_top, pad_bottom)
                    )
    
                return shortcut
            self.align_residual = lambda shortcut, target: align_residual(self, shortcut, target)
            
            W=trainx.shape[3]
            H=trainx.shape[4]
            for i in range(int(self.downnum-1)):
                W=math.floor(((W-self.pool_kernel_size)/self.pool_strides)+1)
                H=math.floor(((H-self.pool_kernel_size)/self.pool_strides)+1)
            if W<1 or H<1:
                print('下采样次数过多,导致最下层图形长度/宽度小于1!')
                return model,predicty,testy,r,p
            if self.pool_strides!=self.pool_kernel_size:
                print('池化步长和池化核不等,将导致下采样前后通道数相除不为整数！')
                return model,predicty,testy,r,p
            for i in range(int(self.downnum)):
                for j in range(int(self.covnum)):
                    if self.total_covnum==0:
                        exec('self.pad'+str(i+1)+'_'+str(j+1)+'=TimeDistributed(ZeroPad2d(padding=(math.floor(math.floor((self.trainx.shape[3]-(self.trainx.shape[3]/self.cov_strides)))/2.0),math.ceil(math.floor((self.trainx.shape[3]-(self.trainx.shape[3]/self.cov_strides)))/2.0),math.floor(math.floor((self.trainx.shape[2]-(self.trainx.shape[2]/self.cov_strides)))/2.0),math.ceil(math.floor((self.trainx.shape[2]-(self.trainx.shape[2]/self.cov_strides)))/2.0))), batch_first=True)', globals(), self.__dict__)
                        exec('self.conv'+str(i+1)+'_'+str(j+1)+'=TimeDistributed(Conv2d(self.trainx.shape[2],self.baselayer,(self.cov_kernelsize,self.cov_kernelsize),stride=self.cov_strides,padding="same"), batch_first=True)', globals(), self.__dict__)
                        if self.if_resnet == 'yes':
                            res_in_channels = int(self.trainx.shape[2])
                            res_out_channels = int(self.baselayer)
                            if res_in_channels != res_out_channels or self.cov_strides != 1:
                                setattr(
                                    self,
                                    'res' + str(i+1) + '_' + str(j+1),
                                    TimeDistributed(
                                        Conv2d(res_in_channels, res_out_channels, (1, 1), stride=self.cov_strides, padding=0),
                                        batch_first=True
                                    )
                                )
                            else:
                                setattr(self, 'res' + str(i+1) + '_' + str(j+1), None)
                        if self.activate=='elu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=ELU()', globals(), self.__dict__)
                        elif self.activate=='leakyrelu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=LeakyReLU()', globals(), self.__dict__)
                        elif self.activate=='prelu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=PReLU()', globals(), self.__dict__)
                        elif self.activate=='relu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=ReLU()', globals(), self.__dict__)
                        elif self.activate=='sigmoid':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Sigmoid()', globals(), self.__dict__)
                        elif self.activate=='tanh':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Tanh()', globals(), self.__dict__)
                        elif self.activate=='softmax':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Softmax()', globals(), self.__dict__)
                        exec('in_channels=self.baselayer', globals(), self.__dict__)
                    else:
                        exec('self.pad'+str(i+1)+'_'+str(j+1)+'=TimeDistributed(ZeroPad2d(padding=(math.floor(math.floor((self.weight'+str(i)+'-(self.weight'+str(i)+'/self.cov_strides)))/2.0),math.ceil(math.floor((self.weight'+str(i)+'-(self.weight'+str(i)+'/self.cov_strides)))/2.0),math.floor(math.floor((self.hight'+str(i)+'-(self.hight'+str(i)+'/self.cov_strides)))/2.0),math.ceil(math.floor((self.hight'+str(i)+'-(self.hight'+str(i)+'/self.cov_strides)))/2.0))), batch_first=True)', globals(), self.__dict__)
                        exec('self.conv'+str(i+1)+'_'+str(j+1)+'=TimeDistributed(Conv2d(self.in_channels,self.baselayer,(self.cov_kernelsize,self.cov_kernelsize),stride=self.cov_strides,padding="same"), batch_first=True)', globals(), self.__dict__)
                        if self.if_resnet == 'yes':
                            res_in_channels = int(self.in_channels)
                            res_out_channels = int(self.baselayer)
                            if res_in_channels != res_out_channels or self.cov_strides != 1:
                                setattr(
                                    self,
                                    'res' + str(i+1) + '_' + str(j+1),
                                    TimeDistributed(
                                        Conv2d(res_in_channels, res_out_channels, (1, 1), stride=self.cov_strides, padding=0),
                                        batch_first=True
                                    )
                                )
                            else:
                                setattr(self, 'res' + str(i+1) + '_' + str(j+1), None)
                        if self.activate=='elu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=ELU()', globals(), self.__dict__)
                        elif self.activate=='leakyrelu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=LeakyReLU()', globals(), self.__dict__)
                        elif self.activate=='prelu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=PReLU()', globals(), self.__dict__)
                        elif self.activate=='relu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=ReLU()', globals(), self.__dict__)
                        elif self.activate=='sigmoid':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Sigmoid()', globals(), self.__dict__)
                        elif self.activate=='tanh':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Tanh()', globals(), self.__dict__)
                        elif self.activate=='softmax':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Softmax()', globals(), self.__dict__)
                        exec('in_channels=self.baselayer', globals(), self.__dict__)
                    self.total_covnum=self.total_covnum+1
                    if self.ifnormalization=='yes':
                        if self.normalization_method=='batchnormalization':
                            exec('self.norm'+str(i+1)+'_'+str(j+1)+'=TimeDistributed(BatchNorm2d(in_channels), batch_first=True)', globals(), self.__dict__)
                        elif self.normalization_method=='layernormalization':
                            exec('self.norm'+str(i+1)+'_'+str(j+1)+'=LayerNorm(in_channels)', globals(), self.__dict__)
                    da_layer_id = i + 1
                    da_channels = int(self.in_channels)
                    da_h = int(getattr(self, 'hight' + str(i)))
                    da_w = int(getattr(self, 'weight' + str(i)))
                    da_time_dim = da_channels * da_h * da_w
                if self.if_bottom_attention_only == 'no' or i == int(self.downnum) - 1:

                    da_layer_id = i + 1

                    # current encoder level final feature shape:
                    # (B, T, C, H, W)
                    #
                    # C is current in_channels after the last conv of this level.
                    # H/W are hight{i}/weight{i} before pooling.
                    da_channels = int(self.in_channels)
                    da_h = int(getattr(self, 'hight' + str(i)))
                    da_w = int(getattr(self, 'weight' + str(i)))

                    # time transformer:
                    # token dimension = T
                    # feature dimension = C * H * W
                    da_time_dim = da_channels * da_h * da_w

                    if da_time_dim % self.num_heads != 0:
                        raise ValueError(
                            "时间 Transformer 的 d_model=C*H*W 必须能被 num_heads 整除。"
                            f" 当前层={da_layer_id}, C={da_channels}, H={da_h}, W={da_w}, "
                            f"C*H*W={da_time_dim}, num_heads={self.num_heads}"
                        )

                    # space transformer:
                    # token dimension = H * W
                    # feature dimension = C
                    if da_channels % self.num_heads != 0:
                        raise ValueError(
                            "空间 Transformer 的 d_model=C 必须能被 num_heads 整除。"
                            f" 当前层={da_layer_id}, C={da_channels}, num_heads={self.num_heads}"
                        )

                    setattr(
                        self,
                        'time_transformer' + str(da_layer_id),
                        TransformerEncoder(
                            TransformerEncoderLayer(
                                da_time_dim,
                                self.num_heads,
                                dim_feedforward=self.dim_feedforward,
                                dropout=self.trans_dropout_rate,
                                activation=self.nlp_activate,
                                batch_first=True
                            ),
                            self.trans_num
                        )
                    )

                    setattr(
                        self,
                        'space_transformer' + str(da_layer_id),
                        TransformerEncoder(
                            TransformerEncoderLayer(
                                da_channels,
                                self.num_heads,
                                dim_feedforward=self.dim_feedforward,
                                dropout=self.trans_dropout_rate,
                                activation=self.nlp_activate,
                                batch_first=True
                            ),
                            self.trans_num
                        )
                    )

                    setattr(
                        self,
                        'time_embed' + str(da_layer_id),
                        nn.Parameter(torch.zeros(1, self.timestep, da_time_dim))
                    )
                    nn.init.trunc_normal_(
                        getattr(self, 'time_embed' + str(da_layer_id)),
                        std=0.02
                    )

                    setattr(
                        self,
                        'space_embed' + str(da_layer_id),
                        nn.Parameter(torch.zeros(1, da_h * da_w, da_channels))
                    )
                    nn.init.trunc_normal_(
                        getattr(self, 'space_embed' + str(da_layer_id)),
                        std=0.02
                    )

                    # learnable timestep aggregation:
                    # input:  (B, C*H*W, T)
                    # output: (B, C*H*W, 1)
                    setattr(
                        self,
                        'time_pool' + str(da_layer_id),
                        Linear(self.timestep, 1)
                    )
                self.k=self.k+1
                if i!=(self.downnum-1):
                    self.baselayer=int(self.baselayer*self.layer_time)
                if self.pool_method=='maxpooling' and i!=(self.downnum-1):
                    exec('self.hight'+str(i+1)+'=1+math.floor((self.hight'+str(i)+'-self.pool_kernel_size)/self.pool_strides)', globals(), self.__dict__)
                    exec('self.weight'+str(i+1)+'=1+math.floor((self.weight'+str(i)+'-self.pool_kernel_size)/self.pool_strides)', globals(), self.__dict__)
                    exec('self.pool'+str(i+1)+'=TimeDistributed(MaxPool2d((self.pool_kernel_size,self.pool_kernel_size),stride=self.pool_strides), batch_first=True)', globals(), self.__dict__)
                elif self.pool_method=='avepooling' and i!=(self.downnum-1):
                    exec('self.hight'+str(i+1)+'=1+math.floor((self.hight'+str(i)+'-self.pool_kernel_size)/self.pool_strides)', globals(), self.__dict__)
                    exec('self.weight'+str(i+1)+'=1+math.floor((self.weight'+str(i)+'-self.pool_kernel_size)/self.pool_strides)', globals(), self.__dict__)
                    exec('self.pool'+str(i+1)+'=TimeDistributed(AvgPool2d((self.pool_kernel_size,self.pool_kernel_size),stride=self.pool_strides), batch_first=True)', globals(), self.__dict__)
            self.l=self.k-1
            self.k=self.k+1
            for m in range(self.downnum,2*self.downnum-1):
                self.baselayer=int(self.baselayer/self.layer_time)
                if m==self.downnum:
                    exec('self.pad'+str(m+1)+'=ZeroPad2d(padding=(math.floor(math.floor((self.weight'+str(m-1)+'-(self.weight'+str(m-1)+'/self.cov_strides)))/2.0),math.ceil(math.floor((self.weight'+str(m-1)+'-(self.weight'+str(m-1)+'/self.cov_strides)))/2.0),math.floor(math.floor((self.hight'+str(m-1)+'-(self.hight'+str(m-1)+'/self.cov_strides)))/2.0),math.ceil(math.floor((self.hight'+str(m-1)+'-(self.hight'+str(m-1)+'/self.cov_strides)))/2.0)))', globals(), self.__dict__)
                else:
                    exec('self.pad'+str(m+1)+'=ZeroPad2d(padding=(math.floor(math.floor((self.weight'+str(m)+'-(self.weight'+str(m)+'/self.cov_strides)))/2.0),math.ceil(math.floor((self.weight'+str(m)+'-(self.weight'+str(m)+'/self.cov_strides)))/2.0),math.floor(math.floor((self.hight'+str(m)+'-(self.hight'+str(m)+'/self.cov_strides)))/2.0),math.ceil(math.floor((self.hight'+str(m)+'-(self.hight'+str(m)+'/self.cov_strides)))/2.0)))', globals(), self.__dict__)
                exec('self.conv'+str(m+1)+'=Conv2d(self.in_channels,self.baselayer,(self.cov_kernelsize,self.cov_kernelsize),stride=self.cov_strides,padding="same")', globals(), self.__dict__)
                if self.if_resnet == 'yes':
                    res_in_channels = int(self.in_channels)
                    res_out_channels = int(self.baselayer)
                    if res_in_channels != res_out_channels or self.cov_strides != 1:
                        setattr(
                            self,
                            'res' + str(m+1),
                            Conv2d(res_in_channels, res_out_channels, (1, 1), stride=self.cov_strides, padding=0)
                        )
                    else:
                        setattr(self, 'res' + str(m+1), None)
                if self.activate=='elu':
                    exec('self.act'+str(m+1)+'=ELU()', globals(), self.__dict__)
                elif self.activate=='leakyrelu':
                    exec('self.act'+str(m+1)+'=LeakyReLU()', globals(), self.__dict__)
                elif self.activate=='prelu':
                    exec('self.act'+str(m+1)+'=PReLU()', globals(), self.__dict__)
                elif self.activate=='relu':
                    exec('self.act'+str(m+1)+'=ReLU()', globals(), self.__dict__)
                elif self.activate=='sigmoid':
                    exec('self.act'+str(m+1)+'=Sigmoid()', globals(), self.__dict__)
                elif self.activate=='tanh':
                    exec('self.act'+str(m+1)+'=Tanh()', globals(), self.__dict__)
                elif self.activate=='softmax':
                    exec('self.act'+str(m+1)+'=Softmax()', globals(), self.__dict__)
                exec('in_channels=self.baselayer', globals(), self.__dict__)
                if self.ifnormalization=='yes':
                    if self.normalization_method=='batchnormalization':
                        exec('self.norm'+str(m+1)+'=BatchNorm2d(in_channels)', globals(), self.__dict__)
                    elif self.normalization_method=='layernormalization':
                        exec('self.norm'+str(m+1)+'=LayerNorm(in_channels)', globals(), self.__dict__)
                exec('self.hight'+str(m+1)+'=self.hight'+str(self.l)+'*self.upsize', globals(), self.__dict__)
                exec('self.weight'+str(m+1)+'=self.weight'+str(self.l)+'*self.upsize', globals(), self.__dict__)
                exec('self.conc'+str(m+1)+'=torch.concatenate', globals(), self.__dict__)   
                exec('self.uppad'+str(m+1)+'=ZeroPad2d(padding=(math.floor((self.weight'+str(self.l-1)+'-self.weight'+str(self.k)+')/2.0),math.ceil((self.weight'+str(self.l-1)+'-self.weight'+str(self.k)+')/2.0),math.floor((self.hight'+str(self.l-1)+'-self.hight'+str(self.k)+')/2.0),math.ceil((self.hight'+str(self.l-1)+'-self.hight'+str(self.k)+')/2.0)))', globals(), self.__dict__)
                exec('self.up'+str(m+1)+'=Upsample(scale_factor=self.upsize, mode="bilinear")', globals(), self.__dict__)
                exec('in_channels=in_channels*2', globals(), self.__dict__)
                for n in range(int(self.covnum)):
                    exec('self.pad'+str(m+1)+'_'+str(n+1)+'=ZeroPad2d(padding=(math.floor(math.floor((self.weight'+str(m+1)+'-(self.weight'+str(m+1)+'/self.cov_strides)))/2.0),math.ceil(math.floor((self.weight'+str(m+1)+'-(self.weight'+str(m+1)+'/self.cov_strides)))/2.0),math.floor(math.floor((self.hight'+str(m+1)+'-(self.hight'+str(m+1)+'/self.cov_strides)))/2.0),math.ceil(math.floor((self.hight'+str(m+1)+'-(self.hight'+str(m+1)+'/self.cov_strides)))/2.0)))', globals(), self.__dict__)
                    exec('self.conv'+str(m+1)+'_'+str(n+1)+'=Conv2d(self.in_channels,self.baselayer,(self.cov_kernelsize,self.cov_kernelsize),stride=self.cov_strides,padding="same")', globals(), self.__dict__)
                    if self.if_resnet == 'yes':
                        res_in_channels = int(self.in_channels)
                        res_out_channels = int(self.baselayer)
                        if res_in_channels != res_out_channels or self.cov_strides != 1:
                            setattr(
                                self,
                                'res' + str(m+1) + '_' + str(n+1),
                                Conv2d(res_in_channels, res_out_channels, (1, 1), stride=self.cov_strides, padding=0)
                            )
                        else:
                            setattr(self, 'res' + str(m+1) + '_' + str(n+1), None)
                    if self.activate=='elu':
                        exec('self.act'+str(m+1)+'_'+str(n+1)+'=ELU()', globals(), self.__dict__)
                    elif self.activate=='leakyrelu':
                        exec('self.act'+str(m+1)+'_'+str(n+1)+'=LeakyReLU()', globals(), self.__dict__)
                    elif self.activate=='prelu':
                        exec('self.act'+str(m+1)+'_'+str(n+1)+'=PReLU()', globals(), self.__dict__)
                    elif self.activate=='relu':
                        exec('self.act'+str(m+1)+'_'+str(n+1)+'=ReLU()', globals(), self.__dict__)
                    elif self.activate=='sigmoid':
                        exec('self.act'+str(m+1)+'_'+str(n+1)+'=Sigmoid()', globals(), self.__dict__)
                    elif self.activate=='tanh':
                        exec('self.act'+str(m+1)+'_'+str(n+1)+'=Tanh()', globals(), self.__dict__)
                    elif self.activate=='softmax':
                        exec('self.act'+str(m+1)+'_'+str(n+1)+'=Softmax()', globals(), self.__dict__)
                    exec('in_channels=self.baselayer', globals(), self.__dict__)
                    if self.ifnormalization=='yes':
                        if self.normalization_method=='batchnormalization':
                            exec('self.norm'+str(m+1)+'_'+str(n+1)+'=BatchNorm2d(in_channels)', globals(), self.__dict__)
                        elif self.normalization_method=='layernormalization':
                            exec('self.norm'+str(m+1)+'_'+str(n+1)+'=LayerNorm(in_channels)', globals(), self.__dict__)
                self.k=self.k+1
                self.l=self.l-1
            if task_mode=='multi_classify':
                exec('self.convlast=Conv2d(self.in_channels,np.max(trainy)+1,(1,1),stride=1,padding="same")', globals(), self.__dict__)
            else:
                exec('self.convlast=Conv2d(self.in_channels,self.trainy.shape[1],(1,1),stride=1,padding="same")', globals(), self.__dict__)
            if if_last_act!='no':
                if if_last_act=='elu':
                    exec('self.actlast=ELU()', globals(), self.__dict__)
                elif if_last_act=='leakyrelu':
                    exec('self.actlast=LeakyReLU()', globals(), self.__dict__)
                elif if_last_act=='prelu':
                    exec('self.actlast=PReLU()', globals(), self.__dict__)
                elif if_last_act=='relu':
                    exec('self.actlast=ReLU()', globals(), self.__dict__)
                elif if_last_act=='sigmoid':
                    exec('self.actlast=Sigmoid()', globals(), self.__dict__)
                elif if_last_act=='tanh':
                    exec('self.actlast=Tanh()', globals(), self.__dict__)
                elif if_last_act=='softmax':
                    exec('self.actlast=Softmax()', globals(), self.__dict__)
        def da_time_attention(self, x, layer_id):
            """
            x: (B, T, C, H, W)
            时间 Transformer:
            token 维度 = T
            feature 维度 = C*H*W
            输出仍然是 (B, T, C, H, W)
            """
            B, T, C, H, W = x.shape
            x_time = x.reshape(B, T, C * H * W)
            x_time = x_time + getattr(self, 'time_embed' + str(layer_id))
            x_time = getattr(self, 'time_transformer' + str(layer_id))(x_time)
            x_time = x_time.reshape(B, T, C, H, W)
            return x_time

        def da_space_attention(self, x, layer_id):
            """
            x: (B, T, C, H, W)
            空间 Transformer:
            对每个 timestep 独立做空间 self-attention
            token 维度 = H*W
            feature 维度 = C
            输出仍然是 (B, T, C, H, W)
            """
            B, T, C, H, W = x.shape
            x_space = x.permute(0, 1, 3, 4, 2).contiguous()
            x_space = x_space.reshape(B * T, H * W, C)
            x_space = x_space + getattr(self, 'space_embed' + str(layer_id))
            x_space = getattr(self, 'space_transformer' + str(layer_id))(x_space)
            x_space = x_space.reshape(B, T, H, W, C)
            x_space = x_space.permute(0, 1, 4, 2, 3).contiguous()
            return x_space

        def da_time_pooling(self, x, layer_id):
            """
            x: (B, T, C, H, W)
            用可学习 Linear(T -> 1) 聚合 timestep 维度
            输出: (B, C, H, W)
            """
            B, T, C, H, W = x.shape
            x_pool = x.reshape(B, T, C * H * W)
            x_pool = x_pool.permute(0, 2, 1).contiguous()
            x_pool = getattr(self, 'time_pool' + str(layer_id))(x_pool)
            x_pool = x_pool.squeeze(-1)
            x_pool = x_pool.reshape(B, C, H, W)
            return x_pool

        def dual_attention_block(self, x, layer_id):
            """
            x: (B, T, C, H, W)
            输出: (B, C, H, W)
            """
            if self.da_attention_order == 'time_first':
                x = self.da_time_attention(x, layer_id)
                x = self.da_space_attention(x, layer_id)
            elif self.da_attention_order == 'space_first':
                x = self.da_space_attention(x, layer_id)
                x = self.da_time_attention(x, layer_id)
            else:
                raise ValueError("da_attention_order must be 'time_first' or 'space_first'")

            x = self.da_time_pooling(x, layer_id)
            return x
        def forward(self, x):
            self.__dict__['x']=x
            self.k=0

            # Encoder
            for i in range(int(self.downnum)):
                for j in range(int(self.covnum)):
                    if i==0 and j==0:
                        exec('model_res_input'+str(i+1)+'_'+str(j+1)+'=x', globals(), self.__dict__)
                    else:
                        if j==0:
                            exec('model_res_input'+str(i+1)+'_'+str(j+1)+'=model_pool'+str(i), globals(), self.__dict__)
                        else:
                            if self.ifnormalization=='yes':
                                exec('model_res_input'+str(i+1)+'_'+str(j+1)+'=model_norm'+str(i+1)+'_'+str(j), globals(), self.__dict__)
                            else:
                                exec('model_res_input'+str(i+1)+'_'+str(j+1)+'=model_act'+str(i+1)+'_'+str(j), globals(), self.__dict__)

                    exec('model_conv'+str(i+1)+'_'+str(j+1)+'=self.conv'+str(i+1)+'_'+str(j+1)+'(model_res_input'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                    exec('model_pad'+str(i+1)+'_'+str(j+1)+'=self.pad'+str(i+1)+'_'+str(j+1)+'(model_conv'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                    exec('model_act'+str(i+1)+'_'+str(j+1)+'=self.act'+str(i+1)+'_'+str(j+1)+'(model_pad'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                    if self.ifnormalization=='yes':
                        exec('model_norm'+str(i+1)+'_'+str(j+1)+'=self.norm'+str(i+1)+'_'+str(j+1)+'(model_act'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)

                    # Encoder residual shortcut
                    if self.if_resnet == 'yes':
                        res_layer = getattr(self, 'res' + str(i+1) + '_' + str(j+1))
                        self.__dict__['res_layer'] = res_layer
                        if res_layer is not None:
                            exec('model_res_shortcut'+str(i+1)+'_'+str(j+1)+'=res_layer(model_res_input'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                        else:
                            exec('model_res_shortcut'+str(i+1)+'_'+str(j+1)+'=model_res_input'+str(i+1)+'_'+str(j+1), globals(), self.__dict__)

                        if self.ifnormalization=='yes':
                            exec('model_res_shortcut'+str(i+1)+'_'+str(j+1)+'=self.align_residual(model_res_shortcut'+str(i+1)+'_'+str(j+1)+', model_norm'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                            exec('model_norm'+str(i+1)+'_'+str(j+1)+'=model_norm'+str(i+1)+'_'+str(j+1)+' + model_res_shortcut'+str(i+1)+'_'+str(j+1), globals(), self.__dict__)
                        else:
                            exec('model_res_shortcut'+str(i+1)+'_'+str(j+1)+'=self.align_residual(model_res_shortcut'+str(i+1)+'_'+str(j+1)+', model_act'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                            exec('model_act'+str(i+1)+'_'+str(j+1)+'=model_act'+str(i+1)+'_'+str(j+1)+' + model_res_shortcut'+str(i+1)+'_'+str(j+1), globals(), self.__dict__)

                # DA skip feature
                #     only bottleneck uses dual_attention_block;
                #     shallow skips use the last timestep.
                #     every skip uses dual_attention_block.
                if self.if_bottom_attention_only == 'yes' and i != int(self.downnum) - 1:
                    if self.ifnormalization=='yes':
                        exec('model_skip'+str(i+1)+'=model_norm'+str(i+1)+'_'+str(j+1)+'[:,-1,:,:,:]', globals(), self.__dict__)
                    else:
                        exec('model_skip'+str(i+1)+'=model_act'+str(i+1)+'_'+str(j+1)+'[:,-1,:,:,:]', globals(), self.__dict__)
                else:
                    if self.ifnormalization=='yes':
                        exec('model_skip'+str(i+1)+'=self.dual_attention_block(model_norm'+str(i+1)+'_'+str(j+1)+','+str(i+1)+')', globals(), self.__dict__)
                    else:
                        exec('model_skip'+str(i+1)+'=self.dual_attention_block(model_act'+str(i+1)+'_'+str(j+1)+','+str(i+1)+')', globals(), self.__dict__)

                self.k=self.k+1

                if i!=(self.downnum-1):
                    if self.ifnormalization=='yes':
                        exec('model_pool'+str(i+1)+'=self.pool'+str(i+1)+'(model_norm'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                    else:
                        exec('model_pool'+str(i+1)+'=self.pool'+str(i+1)+'(model_act'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)

            self.l=self.k-1
            self.k=self.k+1

            # bottom feature after dual attention and learnable timestep pooling
            exec('model_transformer'+str(i+1)+'=model_skip'+str(i+1), globals(), self.__dict__)

            # Decoder
            for m in range(self.downnum,2*self.downnum-1):
                # Decoder main conv input
                if m==self.downnum:
                    exec('model_res_input'+str(m+1)+'=model_transformer'+str(i+1), globals(), self.__dict__)
                else:
                    if self.ifnormalization=='yes':
                        exec('model_res_input'+str(m+1)+'=model_norm'+str(m)+'_'+str(n+1), globals(), self.__dict__)
                    else:
                        exec('model_res_input'+str(m+1)+'=model_act'+str(m)+'_'+str(n+1), globals(), self.__dict__)

                exec('model_conv'+str(m+1)+'=self.conv'+str(m+1)+'(model_res_input'+str(m+1)+')', globals(), self.__dict__)
                exec('model_pad'+str(m+1)+'=self.pad'+str(m+1)+'(model_conv'+str(m+1)+')', globals(), self.__dict__)
                exec('model_act'+str(m+1)+'=self.act'+str(m+1)+'(model_pad'+str(m+1)+')', globals(), self.__dict__)
                if self.ifnormalization=='yes':
                    exec('model_norm'+str(m+1)+'=self.norm'+str(m+1)+'(model_act'+str(m+1)+')', globals(), self.__dict__)

                # Decoder main residual shortcut
                if self.if_resnet == 'yes':
                    res_layer = getattr(self, 'res' + str(m+1))
                    self.__dict__['res_layer'] = res_layer
                    if res_layer is not None:
                        exec('model_res_shortcut'+str(m+1)+'=res_layer(model_res_input'+str(m+1)+')', globals(), self.__dict__)
                    else:
                        exec('model_res_shortcut'+str(m+1)+'=model_res_input'+str(m+1), globals(), self.__dict__)

                    if self.ifnormalization=='yes':
                        exec('model_res_shortcut'+str(m+1)+'=self.align_residual(model_res_shortcut'+str(m+1)+', model_norm'+str(m+1)+')', globals(), self.__dict__)
                        exec('model_norm'+str(m+1)+'=model_norm'+str(m+1)+' + model_res_shortcut'+str(m+1), globals(), self.__dict__)
                    else:
                        exec('model_res_shortcut'+str(m+1)+'=self.align_residual(model_res_shortcut'+str(m+1)+', model_act'+str(m+1)+')', globals(), self.__dict__)
                        exec('model_act'+str(m+1)+'=model_act'+str(m+1)+' + model_res_shortcut'+str(m+1), globals(), self.__dict__)
                        # original decoder uses model_conv{self.k} as covup when no normalization.
                        # Overwrite it so the residual path really affects the following upsample.
                        exec('model_conv'+str(m+1)+'=model_act'+str(m+1), globals(), self.__dict__)

                if self.ifnormalization=='yes':
                    exec('self.covup'+str(m+1)+'=model_norm'+str(self.k), globals(), self.__dict__)
                else:
                    exec('self.covup'+str(m+1)+'=model_conv'+str(self.k), globals(), self.__dict__)

                # DA skip has already been converted to 4D: (B, C, H, W)
                exec('self.covdown'+str(m+1)+'=model_skip'+str(self.l), globals(), self.__dict__)

                exec('model_up'+str(m+1)+'=self.up'+str(m+1)+'(self.covup'+str(m+1)+')', globals(), self.__dict__)
                exec('model_uppad'+str(m+1)+'=self.uppad'+str(m+1)+'(model_up'+str(m+1)+')', globals(), self.__dict__)
                exec('model_conc'+str(m+1)+'=self.conc'+str(m+1)+'((model_uppad'+str(m+1)+',self.covdown'+str(m+1)+'),axis=1)', globals(), self.__dict__)

                for n in range(int(self.covnum)):
                    if n==0:
                        exec('model_res_input'+str(m+1)+'_'+str(n+1)+'=model_conc'+str(m+1), globals(), self.__dict__)
                    else:
                        if self.ifnormalization=='yes':
                            exec('model_res_input'+str(m+1)+'_'+str(n+1)+'=model_norm'+str(m+1)+'_'+str(n), globals(), self.__dict__)
                        else:
                            exec('model_res_input'+str(m+1)+'_'+str(n+1)+'=model_act'+str(m+1)+'_'+str(n), globals(), self.__dict__)

                    exec('model_conv'+str(m+1)+'_'+str(n+1)+'=self.conv'+str(m+1)+'_'+str(n+1)+'(model_res_input'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)
                    exec('model_pad'+str(m+1)+'_'+str(n+1)+'=self.pad'+str(m+1)+'_'+str(n+1)+'(model_conv'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)
                    exec('model_act'+str(m+1)+'_'+str(n+1)+'=self.act'+str(m+1)+'_'+str(n+1)+'(model_pad'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)
                    if self.ifnormalization=='yes':
                        exec('model_norm'+str(m+1)+'_'+str(n+1)+'=self.norm'+str(m+1)+'_'+str(n+1)+'(model_act'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)

                    # Decoder post-concat residual shortcut
                    if self.if_resnet == 'yes':
                        res_layer = getattr(self, 'res' + str(m+1) + '_' + str(n+1))
                        self.__dict__['res_layer'] = res_layer
                        if res_layer is not None:
                            exec('model_res_shortcut'+str(m+1)+'_'+str(n+1)+'=res_layer(model_res_input'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)
                        else:
                            exec('model_res_shortcut'+str(m+1)+'_'+str(n+1)+'=model_res_input'+str(m+1)+'_'+str(n+1), globals(), self.__dict__)

                        if self.ifnormalization=='yes':
                            exec('model_res_shortcut'+str(m+1)+'_'+str(n+1)+'=self.align_residual(model_res_shortcut'+str(m+1)+'_'+str(n+1)+', model_norm'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)
                            exec('model_norm'+str(m+1)+'_'+str(n+1)+'=model_norm'+str(m+1)+'_'+str(n+1)+' + model_res_shortcut'+str(m+1)+'_'+str(n+1), globals(), self.__dict__)
                        else:
                            exec('model_res_shortcut'+str(m+1)+'_'+str(n+1)+'=self.align_residual(model_res_shortcut'+str(m+1)+'_'+str(n+1)+', model_act'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)
                            exec('model_act'+str(m+1)+'_'+str(n+1)+'=model_act'+str(m+1)+'_'+str(n+1)+' + model_res_shortcut'+str(m+1)+'_'+str(n+1), globals(), self.__dict__)

                self.k=self.k+1
                self.l=self.l-1

            if if_last_act!='no':
                if self.ifnormalization=='yes':
                    exec('model_convlast=self.convlast(model_norm'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)
                else:
                    exec('model_convlast=self.convlast(model_act'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)
                outputs=eval('self.actlast(model_convlast)', globals(), self.__dict__)
            else:
                if self.ifnormalization=='yes':
                    outputs=eval('self.convlast(model_norm'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)
                else:
                    outputs=eval('self.convlast(model_act'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)
            return outputs
    if k_fold!=None: 
        models=[]
        for i in range(k_fold):
            models.append(Model(np.asarray(trainx[0:2]),trainy[0:2],timestep,downnum,covnum,layer_time,baselayer,cov_kernelsize,cov_strides,pool_method,pool_kernel_size,pool_strides,ifnormalization,normalization_method,trans_num,num_heads,dim_feedforward,trans_dropout_rate,activate,nlp_activate,da_attention_order,if_bottom_attention_only,if_resnet))
    else:
        model=Model(np.asarray(trainx[0:2]),trainy[0:2],timestep,downnum,covnum,layer_time,baselayer,cov_kernelsize,cov_strides,pool_method,pool_kernel_size,pool_strides,ifnormalization,normalization_method,trans_num,num_heads,dim_feedforward,trans_dropout_rate,activate,nlp_activate,da_attention_order,if_bottom_attention_only,if_resnet)
    if if_best_mode!='no':
        if k_fold!=None:
            for i in range(k_fold):
                models[i].load_state_dict(torch.load(modelpath+'_'+str(i+1)+'.pth',map_location='cpu',weights_only=True))
        else:
            model.load_state_dict(torch.load(modelpath+'.pth',map_location='cpu',weights_only=True))
    if k_fold!=None:
        for i in range(k_fold):
            models[i].to(devices)
    else:
        model.to(devices)
    if k_fold!=None:
        opts=[]
        for i in range(k_fold):
            if optimizer == 'SGD':
                opts.append(SGD(models[i].parameters(), lr=learning_rate))
            elif optimizer == 'Adam':
                opts.append(Adam(models[i].parameters(), lr=learning_rate))
            elif optimizer == 'Nadam':
                opts.append(NAdam(models[i].parameters(), lr=learning_rate))
    else:
        if optimizer == 'SGD':
            opt = SGD(model.parameters(), lr=learning_rate)
        elif optimizer == 'Adam':
            opt = Adam(model.parameters(), lr=learning_rate)
        elif optimizer == 'Nadam':
            opt = NAdam(model.parameters(), lr=learning_rate)
    if if_best_mode!='no' and ifrandom_split!='all_test':
        if k_fold!=None:
            for i in range(k_fold):
                opts[i].load_state_dict(torch.load(modelpath+'_'+str(i+1)+'_opt.pth',map_location='cpu',weights_only=True))
        else:
            opt.load_state_dict(torch.load(modelpath+'_opt.pth',map_location='cpu',weights_only=True))
    if if_print_model=='yes':
        if k_fold!=None:
            print(models[0])
        else:
            print(model)
    if ifrandom_split!='just_model':
        if window_provider is None:
            trainx=np.nan_to_num(trainx,nan=0)
            testx=np.nan_to_num(testx,nan=0)
        if ifrandom_split!='all_test':
            if valid_size!=None or k_fold !=None:
                if k_fold!=None:
                    kf = KFold(n_splits=k_fold, shuffle=True, random_state=25)
                    for fold_no, (train_idx, val_idx) in enumerate(kf.split(trainx, trainy)):
                        X_train_fold, y_train_fold = trainx[train_idx], trainy[train_idx]
                        X_val_fold, y_val_fold = trainx[val_idx], trainy[val_idx]
                        if fold_no==0:
                            train_loss=np.zeros((k_fold,int(np.ceil(X_train_fold.shape[0]/batch_size))))
                            train_metric=np.zeros((k_fold,int(np.ceil(X_train_fold.shape[0]/batch_size))))
                            test_loss=np.zeros((k_fold,int(np.ceil(X_val_fold.shape[0]/batch_size))))
                            test_metric=np.zeros((k_fold,int(np.ceil(X_val_fold.shape[0]/batch_size))))
                        if if_early_stopping!=None:
                            early_stopping = EarlyStopping(patience=if_early_stopping)
                        for i in range(epochs):
                            start = datetime.datetime.now()
                            for j in range(int(np.ceil(X_train_fold.shape[0]/batch_size))):
                                if j == int(X_train_fold.shape[0]/batch_size) :
                                    train_output = models[fold_no](torch.tensor(np.asarray(X_train_fold[j*batch_size:]),dtype=torch.float32,device=devices))
                                    if task_mode=='multi_classify':
                                        if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                            train_losss = torch.mean(loss(train_output, torch.tensor(y_train_fold[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        elif loss_function=='NLLLoss':
                                            train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(y_train_fold[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                            train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(y_train_fold[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='CrossEntropyLoss':
                                            train_metrics = torch.mean(metric(train_output, torch.tensor(y_train_fold[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='NLLLoss':
                                            train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(y_train_fold[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    else:
                                        train_losss = torch.mean(loss(train_output, torch.tensor(y_train_fold[j*batch_size:],dtype=torch.float32,device=devices)))
                                        train_metrics = torch.mean(metric(train_output, torch.tensor(y_train_fold[j*batch_size:],dtype=torch.float32,device=devices)))
                                else:
                                    train_output = models[fold_no](torch.tensor(np.asarray(X_train_fold[j*batch_size:(j+1)*batch_size]),dtype=torch.float32,device=devices))
                                    if task_mode=='multi_classify':
                                        if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                            train_losss = torch.mean(loss(train_output, torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        elif loss_function=='NLLLoss':
                                            train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                            train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='CrossEntropyLoss':
                                            train_metrics = torch.mean(metric(train_output, torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='NLLLoss':
                                            train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    else:
                                        train_losss = torch.mean(loss(train_output, torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                                        train_metrics = torch.mean(metric(train_output, torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                                train_loss[fold_no,j]=np.array(train_losss.item())
                                train_metric[fold_no,j]=np.array(train_metrics.item())
                                opts[fold_no].zero_grad()
                                train_losss.backward(retain_graph=True)
                                opts[fold_no].step()
                            with torch.no_grad():
                                for k in range(int(np.ceil(X_val_fold.shape[0]/batch_size))):
                                    if k == int(X_val_fold.shape[0]/batch_size):
                                        test_output = models[fold_no](torch.tensor(np.asarray(X_val_fold[k*batch_size:]),dtype=torch.float32,device=devices))
                                        if task_mode=='multi_classify':
                                            if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                                test_losss= torch.mean(loss(test_output,torch.tensor(y_val_fold[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                            elif loss_function=='NLLLoss':
                                                test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(y_val_fold[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                            if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                                test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(y_val_fold[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                            elif metrics=='CrossEntropyLoss':
                                                test_metrics=torch.mean(metric(test_output,torch.tensor(y_val_fold[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                            elif metrics=='NLLLoss':
                                                test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(y_val_fold[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        else:
                                            test_losss= torch.mean(loss(test_output,torch.tensor(y_val_fold[k*batch_size:],dtype=torch.float32,device=devices)))
                                            test_metrics=torch.mean(metric(test_output,torch.tensor(y_val_fold[k*batch_size:],dtype=torch.float32,device=devices)))
                                    else:
                                        test_output = models[fold_no](torch.tensor(np.asarray(X_val_fold[k*batch_size:(k+1)*batch_size]),dtype=torch.float32,device=devices))
                                        if task_mode=='multi_classify':
                                            if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                                test_losss= torch.mean(loss(test_output,torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                            elif loss_function=='NLLLoss':
                                                test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                            if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                                test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                            elif metrics=='CrossEntropyLoss':
                                                test_metrics=torch.mean(metric(test_output,torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                            elif metrics=='NLLLoss':
                                                test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        else:
                                            test_losss= torch.mean(loss(test_output,torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                                            test_metrics=torch.mean(metric(test_output,torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                                    test_loss[fold_no,k]=np.array(test_losss.item())
                                    test_metric[fold_no,k]=np.array(test_metrics.item())
                            if if_early_stopping!=None:
                                early_stopping(np.nanmean(test_loss[fold_no,:]), models[fold_no])
                                if early_stopping.early_stop:
                                    early_stopping.load_best_checkpoint(models[fold_no])
                                    break
                            end = datetime.datetime.now()
                            print('第',i+1,'次训练loss:',np.nanmean(train_loss[fold_no,:]),'，metric:',np.nanmean(train_metric[fold_no,:]),'  第',i+1,'次测试loss:',np.nanmean(test_loss[fold_no,:]),'，metric：',np.nanmean(test_metric[fold_no,:]),'训练用时:',end - start)
                else:
                    if if_early_stopping!=None:
                        early_stopping = EarlyStopping(patience=if_early_stopping)
                    for i in range(epochs):
                        start = datetime.datetime.now()
                        if i==0:
                            if ifrandom_split=='yes':
                                trainy,validy,trainx,validx = train_test_split(trainy,trainx,test_size=valid_size/(1-test_size),random_state=25)
                            else:
                                index=_explicit_train_count if _explicit_train_count is not None else int((1-valid_size/(1-test_size))*trainy.shape[0])
                                validy=trainy[index:]
                                trainy=trainy[:index]
                                validx=trainx[index:]
                                trainx=trainx[:index]
                        train_loss=np.zeros((int(np.ceil(trainx.shape[0]/batch_size))))
                        train_metric=np.zeros((int(np.ceil(trainx.shape[0]/batch_size))))
                        test_loss=np.zeros((int(np.ceil(validx.shape[0]/batch_size))))
                        test_metric=np.zeros((int(np.ceil(validx.shape[0]/batch_size))))
                        for j in range(int(np.ceil(trainx.shape[0]/batch_size))):
                            if j == int(trainx.shape[0]/batch_size) :
                                train_output = model(torch.tensor(np.asarray(trainx[j*batch_size:]),dtype=torch.float32,device=devices))
                                if task_mode=='multi_classify':
                                    if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                        train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif loss_function=='NLLLoss':
                                        train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                        train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='CrossEntropyLoss':
                                        train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='NLLLoss':
                                        train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                else:
                                    train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.float32,device=devices)))
                                    train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.float32,device=devices)))
                            else:
                                train_output = model(torch.tensor(np.asarray(trainx[j*batch_size:(j+1)*batch_size]),dtype=torch.float32,device=devices))
                                if task_mode=='multi_classify':
                                    if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                        train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif loss_function=='NLLLoss':
                                        train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                        train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='CrossEntropyLoss':
                                        train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='NLLLoss':
                                        train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                else:
                                    train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                                    train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                            train_loss[j]=np.array(train_losss.item())
                            train_metric[j]=np.array(train_metrics.item())
                            opt.zero_grad()
                            train_losss.backward(retain_graph=True)
                            opt.step() 
                        with torch.no_grad():
                            for k in range(int(np.ceil(validx.shape[0]/batch_size))):
                                if k == int(validx.shape[0]/batch_size) :
                                    test_output = model(torch.tensor(np.asarray(validx[k*batch_size:]),dtype=torch.float32,device=devices))
                                    if task_mode=='multi_classify':
                                        if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                            test_losss= torch.mean(loss(test_output,torch.tensor(validy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        elif loss_function=='NLLLoss':
                                            test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(validy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                            test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(validy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='CrossEntropyLoss':
                                            test_metrics=torch.mean(metric(test_output,torch.tensor(validy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='NLLLoss':
                                            test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(validy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    else:
                                        test_losss= torch.mean(loss(test_output,torch.tensor(validy[k*batch_size:],dtype=torch.float32,device=devices)))
                                        test_metrics=torch.mean(metric(test_output,torch.tensor(validy[k*batch_size:],dtype=torch.float32,device=devices)))
                                else:
                                    test_output = model(torch.tensor(np.asarray(validx[k*batch_size:(k+1)*batch_size]),dtype=torch.float32,device=devices))
                                    if task_mode=='multi_classify':
                                        if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                            test_losss= torch.mean(loss(test_output,torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        elif loss_function=='NLLLoss':
                                            test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                            test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='CrossEntropyLoss':
                                            test_metrics=torch.mean(metric(test_output,torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='NLLLoss':
                                            test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    else:
                                        test_losss= torch.mean(loss(test_output,torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                                        test_metrics=torch.mean(metric(test_output,torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                                test_loss[k]=np.array(test_losss.item())
                                test_metric[k]=np.array(test_metrics.item())
                        if if_early_stopping!=None:
                            early_stopping(np.nanmean(test_loss), model)
                            if early_stopping.early_stop:
                                early_stopping.load_best_checkpoint(model)
                                break
                        end = datetime.datetime.now()
                        print('第',i+1,'次训练loss:',np.nanmean(train_loss),'，metric:',np.nanmean(train_metric),'  第',i+1,'次测试loss:',np.nanmean(test_loss),'，metric：',np.nanmean(test_metric),'训练用时:',end - start)
                    if if_early_stopping!=None:
                        early_stopping.load_best_checkpoint(model)
            else:
                if if_early_stopping!=None:
                    early_stopping = EarlyStopping(patience=if_early_stopping)
                for i in range(epochs):
                    start = datetime.datetime.now()
                    train_loss=np.zeros((int(np.ceil(trainx.shape[0]/batch_size))))
                    train_metric=np.zeros((int(np.ceil(trainx.shape[0]/batch_size))))
                    test_loss=np.zeros((int(np.ceil(testx.shape[0]/batch_size))))
                    test_metric=np.zeros((int(np.ceil(testx.shape[0]/batch_size))))
                    for j in range(int(np.ceil(trainx.shape[0]/batch_size))):
                        if j == int(trainx.shape[0]/batch_size):
                            train_output = model(torch.tensor(np.asarray(trainx[j*batch_size:]),dtype=torch.float32,device=devices))
                            if task_mode=='multi_classify':
                                if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                    train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                elif loss_function=='NLLLoss':
                                    train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                    train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                elif metrics=='CrossEntropyLoss':
                                    train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                elif metrics=='NLLLoss':
                                    train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                            else:
                                train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.float32,device=devices)))
                                train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.float32,device=devices)))
                        else:
                            train_output = model(torch.tensor(np.asarray(trainx[j*batch_size:(j+1)*batch_size]),dtype=torch.float32,device=devices))
                            if task_mode=='multi_classify':
                                if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                    train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                elif loss_function=='NLLLoss':
                                    train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                    train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                elif metrics=='CrossEntropyLoss':
                                    train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                elif metrics=='NLLLoss':
                                    train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                            else:
                                train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                                train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                        train_loss[j]=np.array(train_losss.item())
                        train_metric[j]=np.array(train_metrics.item())
                        opt.zero_grad()
                        train_losss.backward(retain_graph=True)
                        opt.step()
                    with torch.no_grad():
                        for k in range(int(np.ceil(testx.shape[0]/batch_size))):
                            if k == int(testx.shape[0]/batch_size) :
                                test_output = model(torch.tensor(np.asarray(testx[k*batch_size:]),dtype=torch.float32,device=devices))
                                if task_mode=='multi_classify':
                                    if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                        test_losss= torch.mean(loss(test_output,torch.tensor(testy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif loss_function=='NLLLoss':
                                        test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(testy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                        test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(testy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='CrossEntropyLoss':
                                        test_metrics=torch.mean(metric(test_output,torch.tensor(testy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='NLLLoss':
                                        test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(testy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                else:
                                    test_losss= torch.mean(loss(test_output,torch.tensor(testy[k*batch_size:],dtype=torch.float32,device=devices)))
                                    test_metrics=torch.mean(metric(test_output,torch.tensor(testy[k*batch_size:],dtype=torch.float32,device=devices)))
                            else:
                                test_output = model(torch.tensor(np.asarray(testx[k*batch_size:(k+1)*batch_size]),dtype=torch.float32,device=devices))
                                if task_mode=='multi_classify':
                                    if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                        test_losss= torch.mean(loss(test_output,torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif loss_function=='NLLLoss':
                                        test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                        test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='CrossEntropyLoss':
                                        test_metrics=torch.mean(metric(test_output,torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='NLLLoss':
                                        test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                else:
                                    test_losss= torch.mean(loss(test_output,torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                                    test_metrics=torch.mean(metric(test_output,torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                            test_loss[k]=np.array(test_losss.item())
                            test_metric[k]=np.array(test_metrics.item())
                    if if_early_stopping!=None:
                        early_stopping(np.nanmean(test_loss), model)
                        if early_stopping.early_stop:
                            early_stopping.load_best_checkpoint(model)
                            break
                    end = datetime.datetime.now()
                    print('第',i+1,'次训练loss:',np.nanmean(train_loss),'，metric:',np.nanmean(train_metric),'  第',i+1,'次测试loss:',np.nanmean(test_loss),'，metric：',np.nanmean(test_metric),'训练用时:',end - start)
        if k_fold!=None:
            if task_mode=='multi_classify':
                predicty=np.zeros((k_fold,testy.shape[0],int(np.max(vy))+1,testy.shape[2],testy.shape[3]))
            else:
                predicty=np.zeros((k_fold,testy.shape[0],testy.shape[1],testy.shape[2],testy.shape[3]))
        else:
            if task_mode=='multi_classify':
                predicty=np.zeros((testy.shape[0],int(np.max(vy))+1,testy.shape[2],testy.shape[3]))
            else:
                predicty=np.zeros((testy.shape[0],testy.shape[1],testy.shape[2],testy.shape[3]))
        if ifrandom_split=='all_test':
            if k_fold!=None:
                for i in range(k_fold):
                    models[i].eval()
            else:
                model.eval()
        with torch.no_grad():
            if k_fold!=None:
                for i in range(k_fold):
                    for k in range(int(np.ceil(testx.shape[0]/batch_size))):
                        if k == int(testx.shape[0]/batch_size) :
                            predicty_batch = models[i](torch.tensor(np.asarray(testx[k*batch_size:]),dtype=torch.float32,device=devices))
                            predicty[i,k*batch_size:]=predicty_batch.cpu().detach().numpy()
                        else:
                            predicty_batch = models[i](torch.tensor(np.asarray(testx[k*batch_size:(k+1)*batch_size]),dtype=torch.float32,device=devices))
                            predicty[i,k*batch_size:(k+1)*batch_size]=predicty_batch.cpu().detach().numpy()
                predicty=np.nanmean(predicty,axis=0)
            else:
                for k in range(int(np.ceil(testx.shape[0]/batch_size))):
                    if k == int(testx.shape[0]/batch_size) :
                        predicty_batch = model(torch.tensor(np.asarray(testx[k*batch_size:]),dtype=torch.float32,device=devices))
                        predicty[k*batch_size:]=predicty_batch.cpu().detach().numpy()
                    else:
                        predicty_batch = model(torch.tensor(np.asarray(testx[k*batch_size:(k+1)*batch_size]),dtype=torch.float32,device=devices))
                        predicty[k*batch_size:(k+1)*batch_size]=predicty_batch.cpu().detach().numpy()
        predicty = np.nan_to_num(predicty,nan=0)
        r=np.zeros((testy.shape[2],testy.shape[3],testy.shape[1]))
        p=np.zeros((testy.shape[2],testy.shape[3],testy.shape[1]))
        f1score=np.zeros((testy.shape[2],testy.shape[3],testy.shape[1]))
        accuracyscore=np.zeros((testy.shape[2],testy.shape[3],testy.shape[1]))
        recallscore=np.zeros((testy.shape[2],testy.shape[3],testy.shape[1]))
        precisionscore=np.zeros((testy.shape[2],testy.shape[3],testy.shape[1]))
        for i in range(testy.shape[2]):
            for j in range(testy.shape[3]):
                for k in range(testy.shape[1]):
                    if task_mode=='regression':
                        r[i,j,k],p[i,j,k] = pearsonr(predicty[:,k,i,j],testy[:,k,i,j])
                        r[i,j,k]=np.nan_to_num(r[i,j,k],nan=0)
                    elif task_mode=='binary_classify':
                        predicty[:,k,i,j]=[int(round(predicty[l,k,i,j],0)) for l in range(predicty.shape[0])]
                        if metrics=='Recall':
                            r[i,j,k]=recall_score(testy[:,k,i,j], predicty[:,k,i,j])
                        elif metrics=='Precision':
                            r[i,j,k]=precision_score(testy[:,k,i,j], predicty[:,k,i,j])
                        else:
                            r[i,j,k]=accuracy_score(testy[:,k,i,j], predicty[:,k,i,j])
                        p[i,j,k]=0
                        f1score[i,j,k]=f1_score(testy[:,k,i,j],predicty[:,k,i,j])
                        accuracyscore[i,j,k]=accuracy_score(testy[:,k,i,j],predicty[:,k,i,j])
                        recallscore[i,j,k]=recall_score(testy[:,k,i,j],predicty[:,k,i,j])
                        precisionscore[i,j,k]=precision_score(testy[:,k,i,j],predicty[:,k,i,j])
                    elif task_mode=='multi_classify':
                        r[i,j,k]=accuracy_score(testy[:,k,i,j], np.argmax(predicty[:,:,i,j],axis=1))
                        p[i,j,k]=0
        if ifmute == 'no':
            if task_mode=='regression':
                print('相关系数',np.nanmean(r))
            elif task_mode=='binary_classify':
                print('召回率+精确率',np.nanmean(f1score),'准确率',np.nanmean(accuracyscore),'召回率',np.nanmean(recallscore),'精确率',np.nanmean(precisionscore))
            elif task_mode=='multi_classify':
                print('准确率',np.nanmean(r))
        should_run_shap_explainer = False
        def _compute_shap_values_for_model_for_channel(model_instance, current_shap_data, x_sample_shape, output_channel_idx, devices):
            use_gradient_explainer = True 
    
            wrapped_model = ShapWrapper(model_instance, x_sample_shape, output_channel_idx=output_channel_idx).to(devices)
            wrapped_model.eval()
    
            background_data = current_shap_data
            
            if use_gradient_explainer:
                explainer = shap.GradientExplainer(wrapped_model, background_data)
            else:
                explainer = shap.DeepExplainer(wrapped_model, background_data)
            
            raw_shap_values = explainer.shap_values(current_shap_data)
    
            if isinstance(raw_shap_values, list):
                raw_shap_values = raw_shap_values[0]
    
            shap_x_reshaped = raw_shap_values.reshape(raw_shap_values.shape[0], *x_sample_shape)
            return np.abs(shap_x_reshaped)
        if ifheatmap == 'yes' or ifweight in ['yes', 'shap']:
            should_run_shap_explainer = True
    
        if should_run_shap_explainer:
            current_testx_data = testx
            if testx.shape[0] >= 100:
                index = np.random.randint(0, testx.shape[0], size=100)
                current_testx_data = testx[index, ]
    
            x_sample_shape = current_testx_data.shape[1:] 
    
            flat_current_testx_tensor = torch.tensor(current_testx_data, dtype=torch.float32, device=devices).flatten(start_dim=1)
            
            if task_mode=='multi_classify':
                num_model_outputs = int(np.max(vy))+1
            else:
                num_model_outputs = testy.shape[1] 
    
            shap_values_per_output_channel = []
            for output_ch_idx in range(num_model_outputs):
                if k_fold!=None:
                    fold_shap_results_for_channel = []
                    for model_k_fold in models:
                        model_k_fold.eval()
                        fold_shap_results_for_channel.append(_compute_shap_values_for_model_for_channel(
                            model_k_fold, flat_current_testx_tensor, x_sample_shape, output_ch_idx, devices
                        ))
                    shap_values_per_output_channel.append(np.nanmean(np.stack(fold_shap_results_for_channel, axis=0), axis=0))
                else:
                    model.eval()
                    shap_values_per_output_channel.append(_compute_shap_values_for_model_for_channel(
                        model, flat_current_testx_tensor, x_sample_shape, output_ch_idx, devices
                    ))
            shap_values_aggregated = np.stack(shap_values_per_output_channel, axis=0)
    
            if ifheatmap=='yes' and testx.ndim==5: 
                heatmap_per_output_channel = np.nanmean(shap_values_aggregated, axis=(1, 2)) 
                
                heatmap_output_list = []
                for i in range(num_model_outputs):
                    current_heatmap_slice = heatmap_per_output_channel[i, :, :, :] 
                    current_heatmap_slice = current_heatmap_slice.transpose(1, 2, 0) 
                    heatmap_output_list.append(current_heatmap_slice)
                
                if len(heatmap_output_list) > 0 and all(h.shape == heatmap_output_list[0].shape for h in heatmap_output_list):
                     heatmap = np.stack(heatmap_output_list, axis=0) 
                else:
                     heatmap = heatmap_output_list 
    
            if ifweight in ['yes', 'shap']:
                weights_aggregated_per_channel_feature = np.nanmean(shap_values_aggregated, axis=(1, 2, 4, 5))
                
                input_channel_for_weights = trainx.shape[2] 
                
                weights_final_output = np.zeros((num_model_outputs, input_channel_for_weights))
    
                for i in range(num_model_outputs): 
                    current_output_channel_weights = weights_aggregated_per_channel_feature[i, :] 
                    sum_for_norm = np.nansum(current_output_channel_weights) 
                    
                    if sum_for_norm != 0:
                        weights_final_output[i, :] = (current_output_channel_weights / sum_for_norm) * 100
                    else:
                        weights_final_output[i, :] = 0.0 
                    
                    if ifmute == 'no':
                        for j in range(input_channel_for_weights): 
                            print('预报因子',j+1,'对预报值',i+1,'的贡献：',np.array(weights_final_output[i,j]),'％')
                        print('\n')
                weights = weights_final_output
        
        elif ifweight == 'oob':
            
            weights = np.zeros((testy.shape[1],testx.shape[2])) # C_out x C_in
            weight_more=np.zeros((testy.shape[1],testx.shape[2]))
            for i in tqdm(range(testy.shape[1])): 
                for j in range(testx.shape[2]):
                    testx_new=copy.deepcopy(testx)
                    weight_current_feature_output_pair = [] 
                    for k in range(10): 
                        per=np.random.permutation(testx.shape[0])
                        testx_shuffle=testx[per,:,j,:,:] 
                        testx_new[:,:,j,:,:]=testx_shuffle 
                        
                        predicty_new_agg = None
                        if k_fold!=None:
                            predicty_new_folds = []
                            for o in range(k_fold):
                                predicty_new_batch_list = []
                                for n in range(int(np.ceil(testx.shape[0]/batch_size))):
                                    if n == int(testx.shape[0]/batch_size):
                                        current_batch_x = torch.tensor(testx_new[n*batch_size:],dtype=torch.float32,device=devices)
                                    else:
                                        current_batch_x = torch.tensor(testx_new[n*batch_size:(n+1)*batch_size],dtype=torch.float32,device=devices)
                                    predicty_new_batch = models[o](current_batch_x)
                                    predicty_new_batch_list.append(predicty_new_batch.cpu().detach().numpy())
                                predicty_new_folds.append(np.concatenate(predicty_new_batch_list, axis=0))
                            predicty_new_agg = np.nanmean(np.stack(predicty_new_folds, axis=0), axis=0)
                        else:
                            predicty_new_batch_list = []
                            for n in range(int(np.ceil(testx.shape[0]/batch_size))):
                                if n == int(testx.shape[0]/batch_size):
                                    current_batch_x = torch.tensor(testx_new[n*batch_size:],dtype=torch.float32,device=devices)
                                else:
                                    current_batch_x = torch.tensor(testx_new[n*batch_size:(n+1)*batch_size],dtype=torch.float32,device=devices)
                                predicty_new_batch = model(current_batch_x)
                                predicty_new_batch_list.append(predicty_new_batch.cpu().detach().numpy())
                            predicty_new_agg = np.concatenate(predicty_new_batch_list, axis=0)
    
                        for l in range(testy.shape[2]): 
                            for m in range(testy.shape[3]): 
                                if task_mode=='regression':
                                    current_mse_original = sklearn.metrics.mean_squared_error(testy[:,i,l,m],predicty[:,i,l,m])
                                    current_mse_shuffled = sklearn.metrics.mean_squared_error(testy[:,i,l,m],predicty_new_agg[:,i,l,m])
                                    weight_current_feature_output_pair.append(current_mse_shuffled - current_mse_original)
                                elif task_mode=='multi_classify':
                                    current_logloss_original = sklearn.metrics.log_loss(testy[:,i,l,m],predicty[:,:,l,m])
                                    current_logloss_shuffled = sklearn.metrics.log_loss(testy[:,i,l,m],predicty_new_agg[:,:,l,m])
                                    weight_current_feature_output_pair.append(current_logloss_shuffled - current_logloss_original)
                                else: # binary_classify
                                    current_logloss_original = sklearn.metrics.log_loss(testy[:,i,l,m],predicty[:,i,l,m])
                                    current_logloss_shuffled = sklearn.metrics.log_loss(testy[:,i,l,m],predicty_new_agg[:,i,l,m])
                                    weight_current_feature_output_pair.append(current_logloss_shuffled - current_logloss_original)
                    
                    weight_more[i,j]=np.nanmean(weight_current_feature_output_pair)
    
            for i in range(testy.shape[1]): 
                current_output_channel_weights_oob = weight_more[i,:] 
                sum_for_norm = np.nansum(current_output_channel_weights_oob)
                if sum_for_norm != 0:
                    weights[i,:] = (current_output_channel_weights_oob / sum_for_norm) * 100
                else:
                    weights[i,:] = 0.0
    
                if ifmute == 'no':
                    for j in range(testx.shape[2]): 
                        print('预报因子',j+1,'对预报值',i+1,'的贡献：',np.array(weights[i,j]),'％')
                    print('\n')
    if ifsave=='yes':
        if k_fold!=None:
            for i in range(k_fold):
                torch.save(models[i].state_dict(),savepath+'_'+str(i+1)+'.pth')
                torch.save(opts[i].state_dict(),savepath+'_'+str(i+1)+'_opt.pth')
        else:
            torch.save(model.state_dict(),savepath+'.pth')
            torch.save(opt.state_dict(),savepath+'_opt.pth')
    if k_fold!=None:    
        return models,predicty,testy,r,p,heatmap,weights
    else:
        return model,predicty,testy,r,p,heatmap,weights


In [ ]:
import xarray as xr
import numpy as np
sp,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(Path(os.environ.get("YANGTZE_DATA_DIR","data")) / "Surface-pressure.nc"),'sp','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
st,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(Path(os.environ.get("YANGTZE_DATA_DIR","data")) / "2m-temperature.nc"),'t2m','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
sd,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(Path(os.environ.get("YANGTZE_DATA_DIR","data")) / "2m-dewpoint-temperature.nc"),'d2m','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
u850,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(Path(os.environ.get("YANGTZE_DATA_DIR","data")) / "U-component-of-wind-850hpa.nc"),'u','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
u925,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(Path(os.environ.get("YANGTZE_DATA_DIR","data")) / "U-component-of-wind-925hpa.nc"),'u','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
u1000,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(Path(os.environ.get("YANGTZE_DATA_DIR","data")) / "U-component-of-wind-1000hpa.nc"),'u','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
v850,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(Path(os.environ.get("YANGTZE_DATA_DIR","data")) / "V-component-of-wind-850hpa.nc"),'v','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
v925,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(Path(os.environ.get("YANGTZE_DATA_DIR","data")) / "V-component-of-wind-925hpa.nc"),'v','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
v1000,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(Path(os.environ.get("YANGTZE_DATA_DIR","data")) / "V-component-of-wind-1000hpa.nc"),'v','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
w850,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(Path(os.environ.get("YANGTZE_DATA_DIR","data")) / "Vertical-velocity-850hpa.nc"),'w','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
w925,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(Path(os.environ.get("YANGTZE_DATA_DIR","data")) / "Vertical-velocity-925hpa.nc"),'w','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
w1000,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(Path(os.environ.get("YANGTZE_DATA_DIR","data")) / "Vertical-velocity-1000hpa.nc"),'w','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
w850=w850*-0.1
w925=w925*-0.1
w1000=w1000*-0.1


In [ ]:
h = FORECAST_HOURS
vx = np.zeros((3, u850.shape[0]-h, u850.shape[1], u850.shape[2]), dtype='float32')
vy = np.zeros((9, u850.shape[0]-h, u850.shape[1], u850.shape[2]), dtype='float32')
vx[0, :, :] = sp[:-h, :, :]
vx[1, :, :] = st[:-h, :, :]
vx[2, :, :] = sd[:-h, :, :]
vy[0, :, :, :] = u850[h:, :, :]
vy[1, :, :, :] = u925[h:, :, :]
vy[2, :, :, :] = u1000[h:, :, :]
vy[3, :, :, :] = v850[h:, :, :]
vy[4, :, :, :] = v925[h:, :, :]
vy[5, :, :, :] = v1000[h:, :, :]
vy[6, :, :, :] = w850[h:, :, :]
vy[7, :, :, :] = w925[h:, :, :]
vy[8, :, :, :] = w1000[h:, :, :]
vx = vx.transpose(1, 2, 3, 0)
vy = vy.transpose(1, 2, 3, 0)
RAW_TIMES = rounded_hour_times(times)
if np.any(np.diff(RAW_TIMES) != np.timedelta64(1, 'h')):
    raise ValueError('源数据不是连续逐小时记录，不能直接以数组位移表示预报小时数。')
if len(RAW_TIMES)-h != len(vy) or len(vx) != len(vy):
    raise ValueError('输入、目标和时间长度不匹配。')
N_RAW_PAIRS = len(vy)
N_ALL = N_RAW_PAIRS-TIMESTEP+1
# 保留上传函数的实际切分规则：边界按滑窗前的 len(vy) 计算。
TEST_START = int((1-TEST_SIZE)*N_RAW_PAIRS)
TRAIN_STOP = int((1-VALID_SIZE/(1-TEST_SIZE))*TEST_START)
ALL_TIMES = RAW_TIMES[h+TIMESTEP-1:]
TEST_TIMES = ALL_TIMES[TEST_START:]
N_TEST = len(TEST_TIMES)
if not 0 < TRAIN_STOP < TEST_START < N_ALL:
    raise ValueError('样本数不足以使用当前历史窗口和训练/验证/测试比例。')
lat = np.asarray(lat)
lon = np.asarray(lon)
if vy.shape[1:3] != (len(lat), len(lon)):
    raise ValueError('数据空间轴与纬经度长度不匹配。')
print(f'可预测全集 {N_ALL}；训练 {TRAIN_STOP}；验证 {TEST_START-TRAIN_STOP}；测试 {N_TEST}')
print(f'全集：{ALL_TIMES[0]} → {ALL_TIMES[-1]}')
print(f'测试：{TEST_TIMES[0]} → {TEST_TIMES[-1]}')

# vx/vy 已构造完成，释放重复的源场；保留完整 vx 供两个阶段使用。
del sp, st, sd, u850, u925, u1000, v850, v925, v1000, w850, w925, w1000
gc.collect()


In [ ]:
stage1_provider, vmd_mean, vmd_std, VMD_INFO = load_vmd_window_provider(
    vx, RAW_TIMES, FORECAST_HOURS, TIMESTEP, lat, lon, VMD_SPECS,
    train_stop=TRAIN_STOP, vmd_dir=VMD_DIR, chunk_size=IO_CHUNK,
)
if stage1_provider.shape != (N_ALL, TIMESTEP, 20, len(lat), len(lon)):
    raise ValueError(f'第一阶段窗口维度错误：{stage1_provider.shape}')

# 原DA划分边界和测试集保持不变。仅剔除无法由指定VMD分段完整覆盖的训练/验证窗口。
# all_test仍对全部样本推理；被排除的边界样本不会重新混入任何阶段的训练/验证。
sample_ids = np.arange(N_ALL, dtype=np.int64)
context_codes = stage1_provider.context_codes.copy()
STAGE1_SPLITS = {
    'train': sample_ids[(sample_ids < TRAIN_STOP) & (context_codes == 0)],
    'valid': sample_ids[(sample_ids >= TRAIN_STOP) & (sample_ids < TEST_START) & (context_codes == 1)],
    'test': sample_ids[sample_ids >= TEST_START],
}
if np.any(context_codes[STAGE1_SPLITS['test']] != 2):
    raise ValueError('原测试集有窗口不能完整落在VMD test分段，停止以防改变比较时次。')
for name, indices in STAGE1_SPLITS.items():
    if len(indices) == 0:
        raise ValueError(f'{name} 没有可用样本。')
train_starts = STAGE1_SPLITS['train']
train_inputs = np.unique(train_starts[:, None] + np.arange(TIMESTEP))
train_labels = train_starts + TIMESTEP - 1
vx_mean = np.nanmean(vx[train_inputs], axis=0)
vx_std = np.nanstd(vx[train_inputs], axis=0)
vy_mean = np.nanmean(vy[train_labels], axis=0)
vy_std = np.nanstd(vy[train_labels], axis=0)
if not all(np.isfinite(a).all() for a in (vx_mean, vx_std, vy_mean, vy_std)) or np.any(vx_std <= 0) or np.any(vy_std <= 0):
    raise ValueError("Training normalization contains nonfinite values or zero standard deviation.")
# The provider holds a view of vx. Keep that view synchronized.
vx -= vx_mean
vx /= vx_std
vy -= vy_mean
vy /= vy_std
np.testing.assert_array_equal(ALL_TIMES[STAGE1_SPLITS['test']], TEST_TIMES)
VMD_ALIGNMENT = dict(
    forecast_hours=FORECAST_HOURS,
    original_train_count=TRAIN_STOP, original_valid_count=TEST_START-TRAIN_STOP,
    retained_counts={name: len(idx) for name, idx in STAGE1_SPLITS.items()},
    dropped_train_count=TRAIN_STOP-len(STAGE1_SPLITS['train']),
    dropped_valid_count=TEST_START-TRAIN_STOP-len(STAGE1_SPLITS['valid']),
    dropped_train_times=[str(t) for t in ALL_TIMES[np.setdiff1d(sample_ids[:TRAIN_STOP],STAGE1_SPLITS['train'])]],
    dropped_valid_times=[str(t) for t in ALL_TIMES[np.setdiff1d(sample_ids[TRAIN_STOP:TEST_START],STAGE1_SPLITS['valid'])]],
    test_start_time=str(TEST_TIMES[0]), test_end_time=str(TEST_TIMES[-1]),
    all_count=N_ALL, test_count=N_TEST,
    rule='Retain original target partitions; require complete windows within the assigned VMD split.',
)
print(json.dumps(VMD_ALIGNMENT, ensure_ascii=False, indent=2))
VMD_ALIGNMENT_PATH.write_text(json.dumps(VMD_ALIGNMENT,ensure_ascii=False,indent=2), encoding='utf-8')
np.savez_compressed(
    STAGE1_SCALER_PATH, vx_mean=vx_mean, vx_std=vx_std, vy_mean=vy_mean, vy_std=vy_std,
    vmd_mean=vmd_mean, vmd_std=vmd_std, input_channel_order=np.asarray(VMD_INFO['channel_order']),
    forecast_hours=FORECAST_HOURS, timestep=TIMESTEP, raw_times=RAW_TIMES, all_times=ALL_TIMES,
    test_times=TEST_TIMES, latitude=lat, longitude=lon, level=LEVELS,
    train_indices=STAGE1_SPLITS['train'], valid_indices=STAGE1_SPLITS['valid'],
    test_indices=STAGE1_SPLITS['test'], vmd_context_codes=context_codes,
)
print('第一阶段VMD输入与标准化统计量已准备；原始3通道vx独立保留供第二阶段使用。')


In [ ]:
# 提前构造全部窗口并缓存到系统内存。

import gc
import math
import numpy as np
from tqdm import tqdm


class RAMWindowProvider:
    """预先缓存全部窗口；连续批次直接返回内存切片。"""

    ndim = 5
    dtype = np.dtype(np.float32)

    def __init__(self, provider, chunk_size=256, name='窗口缓存'):
        if (
            isinstance(chunk_size, bool)
            or not isinstance(chunk_size, (int, np.integer))
            or chunk_size < 1
        ):
            raise ValueError('chunk_size 必须为正整数。')

        self.shape = tuple(provider.shape)
        self.timestep = provider.timestep

        if hasattr(provider, 'context_codes'):
            self.context_codes = provider.context_codes.copy()

        needed_gib = math.prod(self.shape) * self.dtype.itemsize / 1024**3
        print(
            f'{name}：{self.shape}，需要 {needed_gib:.2f} GiB 系统内存。',
            flush=True
        )

        self.cache = np.empty(
            self.shape,
            dtype=np.float32,
            order='C'
        )

        try:
            for start in tqdm(
                range(0, self.shape[0], chunk_size),
                desc=name
            ):
                stop = min(start + chunk_size, self.shape[0])

                # 沿用原provider的窗口、分段、清洗及通道排列。
                # 不重新标准化，不改变输入数值。
                indices = np.arange(start, stop, dtype=np.int64)
                self.cache[start:stop] = provider.take(indices)

        except BaseException:
            del self.cache
            raise

        # 不保留原provider引用，使旧VMD数组能够释放。
        print(f'{name}完成；训练将直接从内存缓存取数。', flush=True)

    def take(self, indices):
        indices = _window_indices(indices, self.shape[0])

        if len(indices) == 0:
            return self.cache[:0]

        if len(indices) == 1 or np.all(np.diff(indices) == 1):
            # 连续索引返回切片视图，不再复制、拼接整个CPU批次。
            start = int(indices[0])
            return self.cache[start:start + len(indices)]

        # 非连续索引仍保留原顺序和重复项。
        return self.cache[indices]


# 只控制一次性构建缓存的分块大小，不是训练batch_size。
CACHE_BUILD_BATCH = 256

stage1_provider = RAMWindowProvider(
    stage1_provider,
    chunk_size=CACHE_BUILD_BATCH,
    name='第一阶段20通道窗口缓存'
)

gc.collect()


In [ ]:
def wind_metadata(component, stage):
    attrs = dict(
        forecast_hours=FORECAST_HOURS, history_steps=TIMESTEP,
        stage=stage, source_model=str(BASE_MODEL_STEM)+'.pth',
        source_model_sha256=BASE_CHECKPOINT_SHA256,
        normalization='training_rows_per_grid_cell_and_channel',
        test_size=TEST_SIZE, valid_size=VALID_SIZE,
        first_stage_input_channels=20, vmd_stage='first_stage_only',
        vmd_source_notebook=VMD_SOURCE_NOTEBOOK, vmd_alignment_file=str(VMD_ALIGNMENT_PATH),
    )
    if component == 'w':
        attrs['vertical_velocity_note'] = 'Original notebook convention preserved: saved w = -0.1 * ERA5 omega.'
        attrs['comment'] = 'No further conversion of vertical velocity is applied.'
    return attrs

def check_saved_field(field, expected_times):
    if not np.array_equal(field.time.values, expected_times):
        raise ValueError(f'{field.name}: 保存时次与当前模型的目标时次不一致。')
    for name, expected in [('level', LEVELS), ('latitude', lat), ('longitude', lon)]:
        if field.sizes[name] != len(expected) or not np.allclose(field[name].values, expected, rtol=0, atol=1e-5):
            raise ValueError(f'{field.name}: {name} 坐标不匹配。')

def audit_previous_test_files():
    checked = []
    for component in COMPONENTS:
        for kind in ('testy', 'predicty'):
            path = RESULT_DIR / f'da_transunet_{kind}_{component}_{FORECAST_HOURS}_24year_changjiang.nc'
            if not path.exists():
                continue
            with xr.open_dataset(path) as ds:
                field = read_wind_array(ds, f'{kind}_{component}')
                check_saved_field(field, TEST_TIMES)
            checked.append(path.name)
    print(f'核对了 {len(checked)} 个已有测试 NC 的长度、时次和空间坐标。')
    if not checked:
        print('未发现旧测试 NC；本次按第一阶段函数的原始切分规则确定测试集。')


def train_vmd_first_stage():
    global BASE_CHECKPOINT_SHA256
    outputs = model = prediction = truth = None
    paths = []
    try:
        seed_residual_training(FIRST_STAGE_SEED)
        print('第一阶段：原始3通道 + 17个VMD辅助通道，重新训练无PINN的DA-TransUNet。')
        outputs = Auto_DA_TransUnet_pytorch(
            vy, vx, TIMESTEP, DOWNNUM, COVNUM, **COMMON_KWARGS,
            window_provider=stage1_provider, predefined_split=STAGE1_SPLITS,
            if_best_mode='no', modelpath=None, ifrandom_split='no',
            ifsave='yes', savepath=str(BASE_MODEL_STEM),
        )
        model, prediction, truth = outputs[:3]
        expected = (N_TEST, 9, len(lat), len(lon))
        if prediction.shape != expected or truth.shape != expected:
            raise ValueError(f'第一阶段测试输出维度不正确：{prediction.shape}, {truth.shape}')
        for start in range(0, N_TEST, IO_CHUNK):
            stop = min(start+IO_CHUNK, N_TEST)
            ids = TIMESTEP-1+STAGE1_SPLITS['test'][start:stop]
            np.testing.assert_allclose(truth[start:stop], vy[ids].transpose(0,3,1,2), rtol=1e-6, atol=1e-6)
        BASE_CHECKPOINT_SHA256 = checkpoint_sha256(str(BASE_MODEL_STEM)+'.pth')
        clear_model_memory(model)
        outputs = model = None
        gc.collect()
        clear_model_memory()
        for component, channels in COMPONENTS.items():
            mean = vy_mean[:,:,channels].transpose(2,0,1)
            std = vy_std[:,:,channels].transpose(2,0,1)
            for kind, values in [('testy',truth),('predicty',prediction)]:
                path = write_wind_nc(
                    nc_path(kind, component), f'{kind}_{component}', values[:,channels],
                    TEST_TIMES, lat, lon, LEVELS, mean=mean, std=std,
                    attrs=wind_metadata(component,'vmd_first_stage_test'), chunk_size=IO_CHUNK,
                )
                paths.append(str(path))
                print('已保存：',path)
        info = dict(forecast_hours=FORECAST_HOURS, device=DEVICE, seed=FIRST_STAGE_SEED,
                    model=str(BASE_MODEL_STEM)+'.pth', model_sha256=BASE_CHECKPOINT_SHA256,
                    normalization_file=str(STAGE1_SCALER_PATH), vmd=VMD_INFO,
                    alignment=VMD_ALIGNMENT, training_kwargs=COMMON_KWARGS, test_outputs=paths)
        STAGE1_INFO_PATH.write_text(json.dumps(info,ensure_ascii=False,indent=2),encoding='utf-8')
        return info
    finally:
        if model is not None:
            clear_model_memory(model)
        outputs = model = prediction = truth = None
        gc.collect()
        clear_model_memory()

audit_previous_test_files()
stage1_run_info = train_vmd_first_stage()


In [ ]:
BASE_CHECKPOINT_SHA256 = checkpoint_sha256(str(BASE_MODEL_STEM)+'.pth')
def export_baseline_all():
    outputs = model = prediction = truth = None
    try:
        print('第一阶段：加载刚保存的VMD增强模型，用all_test导出全部可预测目标。')
        with torch.no_grad():
            outputs = Auto_DA_TransUnet_pytorch(
                vy, vx, TIMESTEP, DOWNNUM, COVNUM,
                **COMMON_KWARGS, window_provider=stage1_provider,
                if_best_mode='yes', modelpath=str(BASE_MODEL_STEM),
                ifrandom_split='all_test', ifsave='no', savepath=None,
            )
        model, prediction, truth = outputs[:3]
        expected = (N_ALL, 9, len(lat), len(lon))
        if prediction.shape != expected or truth.shape != expected:
            raise ValueError(f'all_test 输出形状不匹配：{prediction.shape}, {truth.shape}; 预期 {expected}')
        # 分块核对函数确实按顺序返回所有可用目标，避免凭长度猜测时次。
        for start in range(0, N_ALL, IO_CHUNK):
            stop = min(start+IO_CHUNK, N_ALL)
            expected_truth = vy[TIMESTEP-1+start:TIMESTEP-1+stop].transpose(0, 3, 1, 2)
            np.testing.assert_allclose(truth[start:stop], expected_truth, rtol=1e-6, atol=1e-6)
        for component, channels in COMPONENTS.items():
            mean = vy_mean[:, :, channels].transpose(2, 0, 1)
            std = vy_std[:, :, channels].transpose(2, 0, 1)
            for kind, values in [('testy', truth), ('predicty', prediction)]:
                path = write_wind_nc(
                    nc_path(kind, component, '_all'), f'{kind}_{component}',
                    values[:, channels], ALL_TIMES, lat, lon, LEVELS,
                    mean=mean, std=std, attrs=wind_metadata(component, 'vmd_first_stage_all'),
                    chunk_size=IO_CHUNK,
                )
                print('已保存：', path)
    finally:
        if model is not None:
            clear_model_memory(model)
        outputs = model = prediction = truth = None
        gc.collect()
        clear_model_memory()

export_baseline_all()
# 后续从已经写入磁盘的 _all 文件读取标签，而非复用第一阶段输出数组。
del vy, stage1_provider
gc.collect()
clear_model_memory()
print('第一阶段完成，模型引用和 CUDA 缓存已释放；VMD输入缓存已释放；原始3通道 vx 保留。')


In [ ]:
def prepare_residual_inputs_and_labels():
    offset = TIMESTEP-1
    # _all[q] 是原始输入第 q+offset 行发出的 h 小时预报。
    # 按起报时次拼接，不能把预测场的有效时次当作原始 vx 的时次。
    issue_times = ALL_TIMES-np.timedelta64(FORECAST_HOURS, 'h')
    np.testing.assert_array_equal(issue_times, RAW_TIMES[offset:N_RAW_PAIRS])
    inputs = np.empty((N_ALL, len(lat), len(lon), 12), dtype=np.float32)
    labels = np.empty((N_ALL, len(lat), len(lon), 9), dtype=np.float32)
    inputs[:, :, :, :3] = vx[offset:]
    for component, channels in COMPONENTS.items():
        input_channels = slice(3+channels.start, 3+channels.stop)
        with xr.open_dataset(nc_path('testy', component, '_all')) as ds_truth, xr.open_dataset(nc_path('predicty', component, '_all')) as ds_prediction:
            truth = read_wind_array(ds_truth, f'testy_{component}')
            prediction = read_wind_array(ds_prediction, f'predicty_{component}')
            check_saved_field(truth, ALL_TIMES)
            check_saved_field(prediction, ALL_TIMES)
            for start in range(0, N_ALL, IO_CHUNK):
                stop = min(start+IO_CHUNK, N_ALL)
                observed = np.asarray(truth.isel(time=slice(start, stop)).values, dtype=np.float32)
                forecast = np.asarray(prediction.isel(time=slice(start, stop)).values, dtype=np.float32)
                error = observed-forecast
                if not np.isfinite(forecast).all() or not np.isfinite(error).all():
                    raise ValueError(f'{component}: 预测输入或残差标签中出现非有限值。')
                inputs[start:stop, :, :, input_channels] = forecast.transpose(0, 2, 3, 1)
                labels[start:stop, :, :, channels] = error.transpose(0, 2, 3, 1)

    # 真值仅用于标签，新增输入仅来自第一阶段 predicty。
    prediction_inputs = inputs[:, :, :, 3:]
    stage2_ids = np.arange(N_ALL-offset, dtype=np.int64)
    global_target_ids = stage2_ids+offset
    explicit_splits = {
        name: stage2_ids[np.isin(global_target_ids, STAGE1_SPLITS[name])]
        for name in ('train', 'valid', 'test')
    }
    train_inputs = np.unique(explicit_splits['train'][:, None] + np.arange(TIMESTEP))
    train_labels = explicit_splits['train'] + offset
    prediction_input_mean = np.nanmean(prediction_inputs[train_inputs], axis=0)
    prediction_input_std_raw = np.nanstd(prediction_inputs[train_inputs], axis=0)
    residual_mean = np.nanmean(labels[train_labels], axis=0)
    residual_std_raw = np.nanstd(labels[train_labels], axis=0)
    for name, value in [('prediction_input_mean', prediction_input_mean),
                        ('prediction_input_std_raw', prediction_input_std_raw),
                        ('residual_mean', residual_mean), ('residual_std_raw', residual_std_raw)]:
        if not np.isfinite(value).all():
            raise ValueError(f'{name}: 标准化统计量存在非有限值。')
    prediction_input_std = np.where(prediction_input_std_raw <= 1e-12, np.float32(1), prediction_input_std_raw)
    residual_std = np.where(residual_std_raw <= 1e-12, np.float32(1), residual_std_raw)
    for start in range(0, N_ALL, IO_CHUNK):
        stop = min(start+IO_CHUNK, N_ALL)
        prediction_inputs[start:stop] = (prediction_inputs[start:stop]-prediction_input_mean)/prediction_input_std
        labels[start:stop] = (labels[start:stop]-residual_mean)/residual_std

    # 第二阶段还需 6 步窗口，因此舍弃最早 5 个训练目标。
    # 微调切分比例，使验证和测试目标时次与第一阶段完全一致。
    residual_test_start = TEST_START-offset
    residual_train_stop = TRAIN_STOP-offset
    if not 0 < residual_train_stop < residual_test_start < N_ALL-offset:
        raise ValueError('有效样本不足以对齐两阶段的训练、验证和测试边界。')
    # 取整数切分区间的中点，避免浮点舍入让 int(...) 少一行。
    residual_test_size = 1-(residual_test_start+0.5)/N_ALL
    residual_valid_size = (1-residual_test_size)*(1-(residual_train_stop+0.5)/residual_test_start)
    split = dict(test_size=float(residual_test_size), valid_size=float(residual_valid_size),
                 test_start=int(residual_test_start), train_stop=int(residual_train_stop))
    assert int((1-split['test_size'])*len(labels)) == residual_test_start
    assert int((1-split['valid_size']/(1-split['test_size']))*residual_test_start) == residual_train_stop
    stage2_target_times = ALL_TIMES[offset:]
    np.testing.assert_array_equal(stage2_target_times[residual_test_start:], TEST_TIMES)
    np.testing.assert_array_equal(stage2_target_times[residual_train_stop:residual_test_start],
                                  ALL_TIMES[TRAIN_STOP:TEST_START])
    assert len(labels[offset+residual_test_start:]) == N_TEST


    # 第二阶段保留与第一阶段相同的有效验证/测试目标；训练仍舍弃最早T-1个目标。
    # 训练目标及其全部第一阶段预测输入均来自train VMD上下文。
    for lag in range(TIMESTEP):
        input_ids=global_target_ids[explicit_splits['train']]-offset+lag
        if np.any(context_codes[input_ids] != 0):
            raise ValueError('残差训练窗口混入了非train的VMD上下文。')
    np.testing.assert_array_equal(stage2_target_times[explicit_splits['valid']], ALL_TIMES[STAGE1_SPLITS['valid']])
    np.testing.assert_array_equal(stage2_target_times[explicit_splits['test']], TEST_TIMES)

    np.savez_compressed(
        SCALER_PATH,
        actual_train_indices=explicit_splits['train'], actual_valid_indices=explicit_splits['valid'],
        actual_test_indices=explicit_splits['test'], stage1_scaler_file=str(STAGE1_SCALER_PATH),
        vx_mean=vx_mean, vx_std=vx_std, vy_mean=vy_mean, vy_std=vy_std,
        prediction_input_mean=prediction_input_mean,
        prediction_input_std=prediction_input_std, prediction_input_std_raw=prediction_input_std_raw,
        residual_mean=residual_mean, residual_std=residual_std, residual_std_raw=residual_std_raw,
        input_scheme='vx_plus_baseline_prediction',
        input_channels=np.asarray(['sp', 'st', 'sd', 'pred_u850', 'pred_u925', 'pred_u1000',
                                   'pred_v850', 'pred_v925', 'pred_v1000', 'pred_w850', 'pred_w925', 'pred_w1000']),
        forecast_hours=FORECAST_HOURS, timestep=TIMESTEP, effective_history_steps=2*TIMESTEP-1,
        test_size=split['test_size'], valid_size=split['valid_size'],
        test_start=split['test_start'], train_stop=split['train_stop'],
        baseline_test_size=TEST_SIZE, baseline_valid_size=VALID_SIZE,
        baseline_test_start=TEST_START, baseline_train_stop=TRAIN_STOP,
        all_times=ALL_TIMES, input_issue_times=issue_times,
        residual_target_times=stage2_target_times, test_times=TEST_TIMES,
        latitude=lat, longitude=lon, level=LEVELS,
        source_model_sha256=BASE_CHECKPOINT_SHA256,
    )
    print('第二阶段输入（原始 3 通道 + 第一阶段预测 9 通道）：', inputs.shape)
    print('残差标签（滑窗前）：', labels.shape)
    print(f'有效历史输入合计 {2*TIMESTEP-1} 个时次；最早舍弃 {offset} 个训练目标。')
    print('第二阶段名义切分参数：', split)
    print('实际使用样本数量：', {name: len(idx) for name,idx in explicit_splits.items()})
    print(f'验证时次保持一致；测试集长度仍为 {N_TEST}，目标时次完全一致。')
    print('训练集标准化统计量已保存：', SCALER_PATH)
    return inputs, labels, residual_mean, residual_std, split, explicit_splits

vx_residual, vy_residual, residual_mean, residual_std, RESIDUAL_SPLIT, STAGE2_SPLITS = prepare_residual_inputs_and_labels()
RESIDUAL_COMMON_KWARGS = dict(COMMON_KWARGS, test_size=RESIDUAL_SPLIT['test_size'],
                            valid_size=RESIDUAL_SPLIT['valid_size'])
gc.collect()

residual_provider = ArrayWindowProvider(vx_residual, TIMESTEP)
assert residual_provider.shape[2] == 12
residual_provider = RAMWindowProvider(
    residual_provider,
    chunk_size=CACHE_BUILD_BATCH,
    name='第二阶段12通道窗口缓存'
)

gc.collect()


In [ ]:
def train_and_save_residual():
    outputs = model = prediction = residual_truth = None
    corrected = wind_truth = None
    written_paths = []
    try:
        seed_residual_training(RESIDUAL_SEED)
        print('第二阶段：原始 vx + 第一阶段预测作为 12 通道输入；验证与测试目标时次对齐。')
        outputs = Auto_DA_TransUnet_pytorch(
            vy_residual, vx_residual, TIMESTEP, DOWNNUM, COVNUM,
            **RESIDUAL_COMMON_KWARGS,
            window_provider=residual_provider, predefined_split=STAGE2_SPLITS,
            if_best_mode='no', modelpath=None, ifrandom_split='no',
            ifsave='yes', savepath=str(RESIDUAL_MODEL_STEM),
        )
        model, prediction, residual_truth = outputs[:3]
        expected = (N_TEST, 9, len(lat), len(lon))
        if prediction.shape != expected or residual_truth.shape != expected:
            raise ValueError(f'残差测试集形状不匹配：{prediction.shape}, {residual_truth.shape}; 预期 {expected}')
        for start in range(0, N_TEST, IO_CHUNK):
            stop = min(start+IO_CHUNK, N_TEST)
            label_start = TIMESTEP-1+RESIDUAL_SPLIT['test_start']
            expected_truth = vy_residual[label_start+start:label_start+stop].transpose(0, 3, 1, 2)
            np.testing.assert_allclose(residual_truth[start:stop], expected_truth, rtol=1e-6, atol=1e-6)

        # 模型已由原函数保存。后面的反标准化和 NC 写入只使用 CPU 数组。
        clear_model_memory(model)
        outputs = model = residual_truth = None
        gc.collect()
        clear_model_memory()

        for component, channels in COMPONENTS.items():
            mean = residual_mean[:, :, channels].transpose(2, 0, 1)
            std = residual_std[:, :, channels].transpose(2, 0, 1)
            corrected = np.empty((N_TEST, 3, len(lat), len(lon)), dtype=np.float32)
            wind_truth = np.empty_like(corrected)
            with xr.open_dataset(nc_path('testy', component, '_all')) as ds_truth, \
                 xr.open_dataset(nc_path('predicty', component, '_all')) as ds_prediction:
                original_truth = read_wind_array(ds_truth, f'testy_{component}').sel(time=TEST_TIMES)
                baseline_prediction = read_wind_array(ds_prediction, f'predicty_{component}').sel(time=TEST_TIMES)
                check_saved_field(original_truth, TEST_TIMES)
                check_saved_field(baseline_prediction, TEST_TIMES)
                for start in range(0, N_TEST, IO_CHUNK):
                    stop = min(start+IO_CHUNK, N_TEST)
                    # 先将残差还原至原始数值尺度，再与同一目标时次的 backbone 预测相加。
                    residual_physical = np.asarray(prediction[start:stop, channels], dtype=np.float64)*std+mean
                    base = np.asarray(baseline_prediction.isel(time=slice(start, stop)).values, dtype=np.float64)
                    final = base+residual_physical
                    if not np.isfinite(final).all() or np.any(np.abs(final) > np.finfo(np.float32).max):
                        raise ValueError(f'{component}: 修正预测包含非有限值或溢出。')
                    corrected[start:stop] = final.astype(np.float32)
                    wind_truth[start:stop] = original_truth.isel(time=slice(start, stop)).values

            attrs = wind_metadata(component, 'residual_corrected_test')
            attrs.update(
                residual_model=str(RESIDUAL_MODEL_STEM)+'.pth',
                residual_seed=RESIDUAL_SEED, test_count=N_TEST,
                residual_input_scheme='vx_plus_baseline_prediction', residual_input_channels=12,
                effective_history_steps=2*TIMESTEP-1,
                prediction_definition='baseline_prediction + inverse_standardized_residual_prediction',
                truth_definition='original wind truth, not the residual label',
            )
            for kind, values in [('testy', wind_truth), ('predicty', corrected)]:
                path = write_wind_nc(
                    nc_path(kind, component, '_residual'), f'{kind}_{component}',
                    values, TEST_TIMES, lat, lon, LEVELS,
                    attrs=attrs, chunk_size=IO_CHUNK,
                )
                written_paths.append(str(path))
                print('已保存：', path)
            corrected = wind_truth = None
            gc.collect()

        checkpoint = Path(str(RESIDUAL_MODEL_STEM)+'.pth')
        if not checkpoint.exists():
            raise FileNotFoundError(f'残差模型未保存：{checkpoint}')
        info = dict(
            forecast_hours=FORECAST_HOURS, history_steps=TIMESTEP, device=DEVICE,
            residual_seed=RESIDUAL_SEED, baseline_model=str(BASE_MODEL_STEM)+'.pth',
            baseline_sha256=BASE_CHECKPOINT_SHA256,
            residual_model=str(checkpoint), residual_sha256=checkpoint_sha256(checkpoint),
            normalization_file=str(SCALER_PATH),
            normalization_scope='retained training inputs and targets',
            input_scheme='vx_plus_baseline_prediction', input_channels=12,
            input_channel_order=['sp', 'st', 'sd', 'pred_u850', 'pred_u925', 'pred_u1000',
                                 'pred_v850', 'pred_v925', 'pred_v1000', 'pred_w850', 'pred_w925', 'pred_w1000'],
            effective_history_steps=2*TIMESTEP-1,
            input_alignment='raw predictors and baseline forecasts aligned by forecast initialization time',
            baseline_training_predictions='in-sample predictions from the fixed first-stage checkpoint',
            dropped_training_targets=TIMESTEP-1,
            train_count=len(STAGE2_SPLITS['train']),
            valid_count=len(STAGE2_SPLITS['valid']),
            vmd_alignment=VMD_ALIGNMENT, first_stage_run_info=str(STAGE1_INFO_PATH),
            all_count=N_ALL, test_count=N_TEST,
            test_start_time=str(TEST_TIMES[0]), test_end_time=str(TEST_TIMES[-1]),
            test_size=RESIDUAL_SPLIT['test_size'], valid_size=RESIDUAL_SPLIT['valid_size'],
            baseline_test_size=TEST_SIZE, baseline_valid_size=VALID_SIZE,
            outputs=written_paths, training_kwargs=RESIDUAL_COMMON_KWARGS,
        )
        RUN_INFO_PATH.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding='utf-8')
        print(f'完成：两阶段测试集均为 {N_TEST} 个时次，时间坐标完全一致。')
        print('残差模型：', checkpoint)
        print('运行记录：', RUN_INFO_PATH)
        return info
    finally:
        if model is not None:
            clear_model_memory(model)
        outputs = model = prediction = residual_truth = corrected = wind_truth = None
        gc.collect()
        clear_model_memory()

residual_run_info = train_and_save_residual()

del residual_provider, vx_residual, vy_residual
gc.collect()
clear_model_memory()
